# Radiology Reporting Harness - minimal template editing pipeline

**Task.** Turn a telegraphic radiologist dictation into a complete structured
report by *minimally editing* the supplied normal template. The leaderboard
metric (RES, lower is better) rewards template-edit fidelity, not free-form
report writing.

**Approach - structured generation (Approach B/D), fully deterministic.**

```
dictation
   |-- segment .......... sentences, "Bones shows ..." cues, technique/history
   |                      boilerplate, and the dictated summary block
   |-- normalise ........ shorthand expansion + corpus-based spell repair
   |-- split ............ "No acute fracture or dislocation" -> two clauses
   |-- route ............ cue > field label > curated anatomy > template text
   |                      overlap > statistics mined from train reports
   |-- edit template .... replace only the contradicted normal statement,
   |                      keep every untouched field byte-identical
   |-- impression ....... reuse the dictated summary, else condense the
   |                      abnormal findings and close with the template line
   |-- validate ......... negation / laterality / measurements / no invented
                          content / untouched fields / no dropped findings
```

Everything runs offline with the standard library (plus pandas and an optional
rapidfuzz accelerator). **No API keys are used anywhere in this notebook**; an
optional LLM refinement hook is included at the end and is disabled unless an
API key is supplied through an environment variable / Kaggle Secret.

Re-running this notebook top to bottom regenerates `submission.csv` exactly -
no per-case manual editing anywhere in the pipeline.


## 1. Environment and data

In [ ]:
import os, sys, json, csv, subprocess

INPUT_DIR = "/kaggle/input/radiology-reporting-harness"
if not os.path.isdir(INPUT_DIR):
    # local / repository checkout fallback
    for cand in ("data", "../data", "/kaggle/input"):
        if os.path.isdir(cand) and os.path.exists(os.path.join(cand, "train.csv")):
            INPUT_DIR = cand
            break
        if os.path.isdir(cand):
            for sub in sorted(os.listdir(cand)):
                p = os.path.join(cand, sub)
                if os.path.isdir(p) and os.path.exists(os.path.join(p, "train.csv")):
                    INPUT_DIR = p
                    break
print("input dir:", INPUT_DIR, os.listdir(INPUT_DIR)[:8])

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
os.chdir(WORK)
os.makedirs("rrh", exist_ok=True)
open(os.path.join("rrh", "__init__.py"), "w").close()
sys.path.insert(0, os.getcwd())

try:
    import rapidfuzz  # noqa: F401
    print("rapidfuzz available")
except ImportError:
    print("rapidfuzz not available - the pure-python fallback is used (slower, identical output)")


## 2. The pipeline

Each cell below writes one module of the `rrh` package, so the whole implementation is visible in the notebook *and* importable by it.

### 2.1 Text utilities - sentence splitting, similarity, edit distance

In [ ]:
%%writefile rrh/textutil.py
"""Low-level text utilities shared by the whole pipeline.

Everything here is deterministic and dependency-free (stdlib only) so the
pipeline reproduces byte-identically on Kaggle, offline.
"""
from __future__ import annotations

import re
import unicodedata
from difflib import SequenceMatcher
from functools import lru_cache

# ---------------------------------------------------------------- whitespace


def clean_ws(text: str) -> str:
    """Normalise unicode + collapse horizontal whitespace, keep newlines."""
    text = unicodedata.normalize("NFKC", text or "")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace(" ", " ").replace("–", "-").replace("—", "-")
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    return text.strip()


def squash(text: str) -> str:
    """Single-line, single-spaced version of a chunk of text."""
    return re.sub(r"\s+", " ", text or "").strip()


# ---------------------------------------------------------------- key forms

_NONWORD = re.compile(r"[^a-z0-9]+")


def key(text: str) -> str:
    """Aggressive normalisation used for equality / similarity comparisons."""
    return _NONWORD.sub(" ", (text or "").lower()).strip()


STOPWORDS = frozenset(
    """a an and are as at be been but by for from had has have in into is it its
    of on or that the there these this to was were with without which who whom
    within are demonstrates demonstrate shows show reveals reveal seen noted note
    identified appears appear present evident visualized visualised""".split()
)


def content_tokens(text: str) -> list[str]:
    return [t for t in key(text).split() if t not in STOPWORDS and len(t) > 2]


def token_set(text: str) -> frozenset[str]:
    return frozenset(content_tokens(text))


def jaccard(a: frozenset, b: frozenset) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def coverage(sub: frozenset, sup: frozenset) -> float:
    """Fraction of `sub` present in `sup`."""
    if not sub:
        return 0.0
    return len(sub & sup) / len(sub)


@lru_cache(maxsize=200_000)
def ratio(a: str, b: str) -> float:
    """Character-level similarity of two normalised strings."""
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def sim(a: str, b: str) -> float:
    return ratio(key(a), key(b))


# ---------------------------------------------------------------- sentences

_SENT_END = re.compile(r"(?<=[.;!?])\s+")
_DECIMAL = re.compile(r"(?<=\d)\.(?=\d)")
_ABBREVS = (
    "dr", "vs", "approx", "e.g", "i.e", "no", "cf", "etc", "mr", "ms", "st",
    "fig", "ca", "wk", "yr", "mo",
)
# telegraphic dictations frequently drop the full stop between sentences
_SENT_FINAL = (
    "seen|noted|identified|present|intact|maintained|preserved|unremarkable|"
    "normal|negative|patent|visualized|visualised|appreciated|demonstrated|observed"
)
_SENT_START = (
    "The|There|No|Mild|Moderate|Severe|Minimal|Small|Large|Normal|Multiple|"
    "Otherwise|Marked|Trace|Focal|Diffuse|Both|Overall|Findings|Impression|An?|"
    "At|Few|Multilevel|Visualized|Rest|Status|Post|Mildly|Grade"
)
# the space is optional: dictations contain run-ons such as "maintainedMinimal"
_MISSING_STOP = re.compile(rf"\b({_SENT_FINAL}) ?(?=(?:{_SENT_START})\b)")


def split_sentences(text: str) -> list[str]:
    """Split prose into sentences without breaking decimals or level labels."""
    if not text:
        return []
    guarded = _DECIMAL.sub("\x00", text)
    guarded = re.sub(r"\b([A-Z])\.(?=\s*[A-Z]\.)", lambda m: m.group(1) + "\x00", guarded)
    for ab in _ABBREVS:
        guarded = re.sub(
            rf"\b({re.escape(ab)})\.", lambda m: m.group(1) + "\x00", guarded, flags=re.I
        )
    out: list[str] = []
    for line in guarded.split("\n"):
        line = line.strip()
        if not line:
            continue
        for piece in _SENT_END.split(line):
            piece = piece.strip()
            if not piece:
                continue
            piece = _MISSING_STOP.sub(lambda m: m.group(1) + ".\x01", piece)
            for sub in piece.split("\x01"):
                sub = sub.strip()
                if sub:
                    out.append(sub.replace("\x00", "."))
    return out


def cap_first(text: str) -> str:
    """Upper-case the first alphabetic character, leave the rest untouched."""
    for i, ch in enumerate(text):
        if ch.isalpha():
            # do not lower-case ALLCAPS acronyms that already start the string
            return text[:i] + ch.upper() + text[i + 1 :]
        if ch not in "([\"' ":
            break
    return text


def end_period(text: str) -> str:
    text = text.rstrip()
    if not text:
        return text
    if text[-1] in ".;:!?":
        return text[:-1] + "." if text[-1] == ";" else text
    return text + "."


def tidy_sentence(text: str) -> str:
    """Canonical sentence rendering: trimmed, capitalised, single final period."""
    text = squash(text)
    text = re.sub(r"^[\-•*\d]+[\.\)]\s*", "", text)  # strip list bullets
    text = re.sub(r"\s+([,.;:])", r"\1", text)
    text = re.sub(r"\.{2,}", ".", text)
    if not text:
        return ""
    return end_period(cap_first(text))


# ---------------------------------------------------------------- distances


try:  # optional accelerator; the pure-python fallback gives identical results
    from rapidfuzz.distance import Levenshtein as _RF

    def levenshtein(a, b) -> int:
        return _RF.distance(a, b)

except Exception:  # pragma: no cover - Kaggle images ship rapidfuzz, but be safe
    def levenshtein(a, b) -> int:
        return _levenshtein_py(a, b)


def _levenshtein_py(a, b) -> int:
    """Edit distance over any two sequences (str or list of tokens)."""
    if a == b:
        return 0
    la, lb = len(a), len(b)
    if la == 0:
        return lb
    if lb == 0:
        return la
    prev = list(range(lb + 1))
    for i in range(1, la + 1):
        cur = [i] + [0] * lb
        ai = a[i - 1]
        for j in range(1, lb + 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ai != b[j - 1]))
        prev = cur
    return prev[lb]


### 2.2 Radiology lexicon - anatomy->field concepts, shorthand, spell repair

In [ ]:
%%writefile rrh/lexicon.py
"""Curated radiology knowledge used to route findings to template fields.

Kept small and explicit: it encodes *which anatomical field a term belongs to*,
never what a finding means clinically.  Nothing here invents content.
"""
from __future__ import annotations

import re

# canonical field concept -> terms that belong to that field
CONCEPT_TERMS: dict[str, tuple[str, ...]] = {
    "BONES": (
        "bone", "bones", "osseous", "fracture", "marrow", "cortex", "cortical",
        "lytic", "blastic", "sclerotic", "osteopenia", "osteoporosis", "mineralization",
        "vertebral body", "vertebral bodies", "endplate", "spondylosis", "spur",
        "osteophyte", "avulsion", "periosteal", "bone island", "enchondroma",
        "acromion", "tuberosity", "condyle", "malleolus", "calcaneus", "scaphoid",
        "clavicle", "rib", "sternum", "pedicle", "spinous process", "odontoid",
    ),
    "JOINTS": (
        "joint", "joints", "joint space", "articular", "dislocation", "subluxation",
        "arthrosis", "arthritis", "osteoarthrosis", "osteoarthritis", "degenerative change",
        "alignment", "effusion", "facet", "sacroiliac", "acromioclavicular",
        "glenohumeral", "carpometacarpal", "interphalangeal", "metacarpophalangeal",
    ),
    "SOFT TISSUES": (
        "soft tissue", "soft tissues", "swelling", "cellulitis", "edema", "oedema",
        "foreign body", "subcutaneous", "hematoma", "seroma", "abscess", "lipoma",
        "phlebolith", "calcified lymph node", "fat pad", "ganglion",
    ),
    "MUSCLES": (
        "muscle", "muscles", "musculature", "atrophy", "fatty infiltration",
        "myotendinous", "strain", "bulk",
    ),
    "TENDONS": (
        "tendon", "tendons", "tendinosis", "tendinopathy", "tendinitis", "tenosynovitis",
        "supraspinatus", "infraspinatus", "subscapularis", "teres minor", "biceps",
        "achilles", "peroneal", "quadriceps", "patellar tendon", "rotator cuff",
    ),
    "LIGAMENTS": (
        "ligament", "ligaments", "ligamentous", "cruciate", "collateral", "acl", "pcl",
        "mcl", "lcl", "lisfranc", "spring ligament", "deltoid ligament", "retinaculum",
        "syndesmosis", "talofibular",
    ),
    "MENISCI": ("meniscus", "menisci", "meniscal", "bucket-handle", "mucoid degeneration"),
    "CARTILAGE": ("cartilage", "chondral", "chondromalacia", "osteochondral"),
    "BURSAE": ("bursa", "bursae", "bursitis", "subacromial", "subdeltoid", "prepatellar"),
    "NERVES": ("nerve", "nerves", "neuroma", "median nerve", "ulnar nerve", "sciatic"),
    "LUNGS": (
        "lung", "lungs", "pulmonary", "airspace", "air-space", "consolidation",
        "opacity", "infiltrate", "atelectasis", "nodule", "emphysema", "bronchiectasis",
        "interstitial", "edema", "reticular", "ground-glass", "airway", "bronch",
    ),
    "PLEURA": ("pleura", "pleural", "effusion", "pneumothorax", "pleural thickening"),
    "HEART": ("heart", "cardiac", "cardiomegaly", "cardiac silhouette"),
    "MEDIASTINUM": (
        "mediastinum", "mediastinal", "hilum", "hila", "hilar", "aorta", "aortic",
        "great vessel", "trachea",
    ),
    "DIAPHRAGM": ("diaphragm", "diaphragmatic", "hemidiaphragm", "subdiaphragmatic", "free air"),
    "SUPPORT DEVICES": (
        "line", "tube", "catheter", "pacemaker", "device", "stent", "port",
        "endotracheal", "picc", "drain", "hardware", "screw", "plate", "prosthesis",
    ),
    "LIVER": ("liver", "hepatic", "hepatomegaly", "steatosis", "fatty liver"),
    "GALLBLADDER": ("gallbladder", "gallstone", "cholelithiasis", "biliary", "cbd"),
    "KIDNEYS": ("kidney", "kidneys", "renal", "hydronephrosis", "calculus", "nephrolithiasis"),
    "SPLEEN": ("spleen", "splenic", "splenomegaly"),
    "PANCREAS": ("pancreas", "pancreatic"),
    "BLADDER": ("bladder", "urinary bladder", "vesical"),
    "BOWEL": ("bowel", "colon", "small bowel", "ileus", "obstruction", "appendix"),
    "UTERUS": ("uterus", "uterine", "endometrium", "endometrial", "myometrium", "fibroid"),
    "OVARIES": ("ovary", "ovaries", "adnexa", "adnexal", "follicle"),
    "PROSTATE": ("prostate", "prostatic", "seminal vesicle"),
    "BRAIN": (
        "brain", "cerebral", "cerebellum", "parenchyma", "white matter", "gray matter",
        "infarct", "hemorrhage", "midline shift", "ventricle", "ventricular", "gliosis",
        "encephalomalacia", "mass effect",
    ),
    "SPINAL CORD": ("cord", "spinal cord", "myelomalacia", "syrinx", "cord signal"),
    "DISC": (
        "disc", "disk", "herniation", "protrusion", "extrusion", "bulge", "annular",
        "desiccation", "canal stenosis", "foraminal", "thecal sac", "osteophyte complex",
    ),
    "ALIGNMENT": (
        "alignment", "lordosis", "kyphosis", "scoliosis", "spondylolisthesis",
        "listhesis", "curvature", "straightening",
    ),
    "VESSELS": ("artery", "arterial", "vein", "venous", "stenosis", "aneurysm", "thrombus"),
    "SINUSES": ("sinus", "sinuses", "maxillary", "ethmoid", "sphenoid", "frontal sinus", "mucosal",
                "mastoid", "mastoiditis", "mastoid air cells"),
    "BILIARY": ("bile duct", "biliary", "cbd", "common bile duct", "common hepatic duct",
                "choledocholithiasis", "intrahepatic biliary", "biliary radicle",
                "biliary dilatation", "biliary stricture"),
    "PANCREATIC DUCT": ("pancreatic duct", "duct of wirsung", "main pancreatic duct"),
    "URETERS": ("ureter", "ureters", "ureteric", "ureteral", "hydroureter",
                "hydroureteronephrosis", "ureterovesical", "collecting system"),
    "SPINAL CANAL": ("spinal canal", "canal stenosis", "ap diameter", "central canal",
                     "canal diameter", "thecal"),
    "DISC SPACES": ("disc space", "disc height", "intervertebral", "disc space narrowing"),
    "NEURAL FORAMINA": ("neural foramen", "neural foramina", "foraminal", "foramina"),
    "FACET JOINTS": ("facet", "facets", "facet joint", "facet arthropathy", "zygapophyseal"),
    "ORBITS": ("orbit", "orbits", "globe", "optic nerve", "extraocular"),
}

# template label (upper-cased) -> canonical concept
LABEL_ALIASES: dict[str, str] = {
    "BONE": "BONES", "BONES": "BONES", "OSSEOUS STRUCTURES": "BONES",
    "OSSEOUS": "BONES", "BONES AND JOINTS": "BONES", "SKELETAL": "BONES",
    "VERTEBRAL BODIES": "BONES", "VERTEBRAL BODIES AND ALIGNMENT": "BONES",
    "OSSEOUS STRUCTURES AND ALIGNMENT": "BONES", "BONY STRUCTURES": "BONES",
    "JOINT": "JOINTS", "JOINTS": "JOINTS", "JOINT SPACES": "JOINTS",
    "JOINT SPACES AND ALIGNMENT": "JOINTS", "ARTICULATIONS": "JOINTS",
    "SOFT TISSUE": "SOFT TISSUES", "SOFT TISSUES": "SOFT TISSUES",
    "PARAVERTEBRAL SOFT TISSUES": "SOFT TISSUES", "PARASPINAL SOFT TISSUES": "SOFT TISSUES",
    "SURROUNDING SOFT TISSUES": "SOFT TISSUES",
    "MUSCLES": "MUSCLES", "MUSCULATURE": "MUSCLES", "MUSCLE": "MUSCLES",
    "TENDONS": "TENDONS", "TENDON": "TENDONS", "ROTATOR CUFF": "TENDONS",
    "LIGAMENTS": "LIGAMENTS", "LIGAMENT": "LIGAMENTS",
    "MENISCI": "MENISCI", "MENISCUS": "MENISCI",
    "ARTICULAR CARTILAGE": "CARTILAGE", "CARTILAGE": "CARTILAGE",
    "BURSAE": "BURSAE", "BURSA": "BURSAE",
    "NERVES": "NERVES", "NERVE": "NERVES",
    "LUNGS": "LUNGS", "LUNG": "LUNGS", "LUNGS/AIRWAYS": "LUNGS",
    "LUNGS AND AIRWAYS": "LUNGS", "LUNG PARENCHYMA": "LUNGS", "AIRWAYS": "LUNGS",
    "PLEURA": "PLEURA", "PLEURAL SPACES": "PLEURA", "PLEURAL SPACE": "PLEURA",
    "HEART": "HEART", "CARDIAC SILHOUETTE": "HEART", "CARDIOVASCULAR": "HEART",
    "CARDIOMEDIASTINAL SILHOUETTE": "HEART",
    "MEDIASTINUM": "MEDIASTINUM", "MEDIASTINUM/HILA": "MEDIASTINUM",
    "MEDIASTINUM AND HILA": "MEDIASTINUM", "HILA": "MEDIASTINUM",
    "DIAPHRAGM": "DIAPHRAGM", "DIAPHRAGMS": "DIAPHRAGM",
    "SUPPORT DEVICES": "SUPPORT DEVICES", "LINES/TUBES/SUPPORT DEVICES": "SUPPORT DEVICES",
    "LINES AND TUBES": "SUPPORT DEVICES", "DEVICES": "SUPPORT DEVICES",
    "LIVER": "LIVER", "GALLBLADDER": "GALLBLADDER", "GALLBLADDER AND BILIARY": "GALLBLADDER",
    "BILIARY SYSTEM": "GALLBLADDER", "KIDNEYS": "KIDNEYS", "KIDNEY": "KIDNEYS",
    "SPLEEN": "SPLEEN", "PANCREAS": "PANCREAS", "BLADDER": "BLADDER",
    "URINARY BLADDER": "BLADDER", "BOWEL": "BOWEL", "BOWEL GAS PATTERN": "BOWEL",
    "UTERUS": "UTERUS", "OVARIES": "OVARIES", "ADNEXA": "OVARIES", "PROSTATE": "PROSTATE",
    "BRAIN": "BRAIN", "BRAIN PARENCHYMA": "BRAIN", "PARENCHYMA": "BRAIN",
    "SPINAL CORD": "SPINAL CORD", "CORD": "SPINAL CORD",
    "ALIGNMENT": "ALIGNMENT", "SINUSES": "SINUSES", "PARANASAL SINUSES": "SINUSES",
    "ORBITS": "ORBITS", "VESSELS": "VESSELS", "VASCULAR": "VESSELS",
    "BILIARY TREE": "BILIARY", "BILE DUCTS": "BILIARY", "BILIARY": "BILIARY",
    "COMMON BILE DUCT": "BILIARY", "PANCREATIC DUCT": "PANCREATIC DUCT",
    "URETERS": "URETERS", "URETERS AND URINARY BLADDER": "URETERS",
    "SPINAL CANAL": "SPINAL CANAL", "CENTRAL CANAL": "SPINAL CANAL",
    "DISC SPACES": "DISC SPACES", "INTERVERTEBRAL DISCS": "DISC SPACES",
    "DISCS": "DISC SPACES", "NEURAL FORAMINA": "NEURAL FORAMINA",
    "FACET JOINTS": "FACET JOINTS", "SINUSES AND MASTOIDS": "SINUSES",
    "MASTOIDS": "SINUSES", "VERTEBRAE": "BONES", "VERTEBRAL BODIES/ALIGNMENT": "BONES",
}

_STEM_SUFFIXES = ("ies", "ives", "ing", "ous", "als", "ial", "ic", "es", "s", "al", "ar", "y")


def stem(word: str) -> str:
    w = word.lower()
    for suf in _STEM_SUFFIXES:
        if len(w) > len(suf) + 3 and w.endswith(suf):
            return w[: -len(suf)]
    return w


def concept_for_label(label: str) -> str | None:
    lab = re.sub(r"\s+", " ", (label or "").upper()).strip()
    if lab in LABEL_ALIASES:
        return LABEL_ALIASES[lab]
    for alias, concept in LABEL_ALIASES.items():
        if alias in lab:
            return concept
    return None


_TERM_INDEX: dict[str, set[str]] = {}
for _concept, _terms in CONCEPT_TERMS.items():
    for _t in _terms:
        _TERM_INDEX.setdefault(_t.lower(), set()).add(_concept)


def concept_hits(text: str) -> dict[str, int]:
    """How many curated terms of each concept occur in `text`."""
    low = " " + re.sub(r"[^a-z0-9 ]", " ", (text or "").lower()) + " "
    hits: dict[str, int] = {}
    for term, concepts in _TERM_INDEX.items():
        if f" {term} " in low or (" " in term and term in low):
            for c in concepts:
                hits[c] = hits.get(c, 0) + 1
    return hits


# ------------------------------------------------------------------ shorthand
# High-precision expansions of dictation shorthand and recurrent misspellings.
# Only unambiguous entries: nothing here changes clinical meaning.
SHORTHAND: dict[str, str] = {
    "degen": "degenerative",
    "degenrative": "degenerative",
    "degerative": "degenerative",
    "chnges": "changes",
    "chages": "changes",
    "norml": "normal",
    "nomal": "normal",
    "effusuon": "effusion",
    "effusiom": "effusion",
    "cacified": "calcified",
    "calcenal": "calcaneal",
    "lymphnodes": "lymph nodes",
    "lymphnode": "lymph node",
    "osteophyes": "osteophytes",
    "oosteophytes": "osteophytes",
    "trignonum": "trigonum",
    "rt": "right",
    "lt": "left",
    "bilat": "bilateral",
    "jt": "joint",
    "jts": "joints",
    "fx": "fracture",
    "wnl": "within normal limits",
    "w/o": "without",
    "w/": "with",
    "c/w": "consistent with",
    "s/p": "status post",
    "b/l": "bilateral",
    "h/o": "history of",
}

_SHORTHAND_RE = re.compile(
    r"(?<![A-Za-z0-9])(" + "|".join(sorted((re.escape(k) for k in SHORTHAND), key=len, reverse=True))
    + r")(?![A-Za-z0-9])",
    re.I,
)


def normalize_shorthand(text: str) -> str:
    """Expand dictation shorthand so the report reads as prose."""
    if not text:
        return text

    def repl(m: re.Match) -> str:
        src = m.group(1)
        out = SHORTHAND[src.lower()]
        return out.capitalize() if src[:1].isupper() else out

    return _SHORTHAND_RE.sub(repl, text)


# ------------------------------------------------------------- spell repair
_WORD_RE = re.compile(r"[A-Za-z][A-Za-z'\-]{2,}")
_PROTECT = frozenset(
    """mm cm ml cc iv ap pa lat oblique t1 t2 stir flair dwi adc grade type
    lung rads birads""".split()
)


def build_vocabulary(texts) -> dict[str, int]:
    """Vocabulary of words that actually occur in reports/templates."""
    from collections import Counter

    vocab: Counter = Counter()
    for t in texts:
        for w in _WORD_RE.findall((t or "").lower()):
            vocab[w] += 1
    return dict(vocab)


def correct_spelling(text: str, vocab: dict[str, int], min_count: int = 3) -> str:
    """Repair dictation typos against the corpus vocabulary.

    Conservative by construction: only words absent from the vocabulary are
    touched, the replacement must be a frequent corpus word, and the edit
    distance budget scales with word length.
    """
    if not text or not vocab:
        return text
    from rapidfuzz import process, fuzz

    choices = [w for w, c in vocab.items() if c >= min_count]
    if not choices:
        return text
    cache: dict[str, str] = {}

    def repl(m: re.Match) -> str:
        w = m.group(0)
        low = w.lower()
        if low in vocab or low in _PROTECT or len(low) < 5:
            return w
        if low in cache:
            out = cache[low]
        else:
            budget = 1 if len(low) < 8 else 2
            hit = process.extractOne(
                low, choices, scorer=fuzz.ratio, score_cutoff=100 * (1 - budget / len(low))
            )
            out = low
            if hit:
                cand = hit[0]
                if abs(len(cand) - len(low)) <= budget and cand[0] == low[0]:
                    out = cand
            cache[low] = out
        if out == low:
            return w
        return out.capitalize() if w[:1].isupper() else out

    return _WORD_RE.sub(repl, text)


### 2.3 Template parsing and report rendering

In [ ]:
%%writefile rrh/template.py
"""Parsing and rendering of the supplied normal template.

The template is the *starting report*: the renderer therefore reproduces the
template's field labels and field order exactly, only upper-casing labels and
inserting the blank-line separators used by the reference reports.
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field as dc_field

from .textutil import cap_first, clean_ws, split_sentences, squash

HEAD_FINDINGS = re.compile(r"^[ \t]*FINDINGS[ \t]*:[ \t]*", re.I | re.M)
HEAD_IMPRESSION = re.compile(r"^[ \t]*IMPRESSION[ \t]*:[ \t]*", re.I | re.M)

# A label is a short, colon-terminated prefix at the start of a line.
LABEL_RE = re.compile(r"^[ \t]*([A-Za-z][A-Za-z0-9 ,'\-/&\.\(\)]{0,58}?)[ \t]*:[ \t]*(.*)$")

# Labels that the reference reports always leave empty.
ALWAYS_EMPTY = {"OTHER FINDINGS"}


def _is_label(line: str) -> tuple[str, str] | None:
    m = LABEL_RE.match(line)
    if not m:
        return None
    label, rest = m.group(1).strip(), m.group(2).strip()
    if not label or len(label.split()) > 7:
        return None
    # reject prose that merely happens to contain a colon
    if re.search(r"[.!?]\s", label):
        return None
    return label, rest


@dataclass
class Field:
    label_raw: str
    label: str
    text: str
    is_group: bool
    order: int
    is_free: bool = False
    sentences: list[str] = dc_field(default_factory=list)

    def __post_init__(self) -> None:
        if not self.sentences:
            self.sentences = split_sentences(self.text)


@dataclass
class Template:
    raw: str
    fields: list[Field]
    impression: list[str]
    findings_free: list[str]

    @property
    def labels(self) -> list[str]:
        return [f.label for f in self.fields]

    def by_label(self, label: str) -> Field | None:
        for f in self.fields:
            if f.label == label:
                return f
        return None


def parse_template(text: str) -> Template:
    text = clean_ws(text)
    mf = HEAD_FINDINGS.search(text)
    mi = HEAD_IMPRESSION.search(text)
    if mf and mi and mi.start() > mf.start():
        body, imp = text[mf.end() : mi.start()], text[mi.end() :]
    elif mf:
        body, imp = text[mf.end() :], ""
    elif mi:
        body, imp = text[: mi.start()], text[mi.end() :]
    else:
        body, imp = text, ""

    fields: list[Field] = []
    free: list[str] = []
    order = 0
    for line in body.split("\n"):
        line = line.strip()
        if not line:
            continue
        parsed = _is_label(line)
        if parsed:
            label_raw, rest = parsed
            fields.append(
                Field(
                    label_raw=label_raw,
                    label=label_raw.upper(),
                    text=rest,
                    is_group=(rest == ""),
                    order=order,
                )
            )
            order += 1
        else:
            # Prose templates (no labels) keep every line as its own block.
            free.append(line)
            fields.append(
                Field(
                    label_raw="",
                    label="",
                    text=line,
                    is_group=False,
                    order=order,
                    is_free=True,
                )
            )
            order += 1

    impression = [ln.strip() for ln in imp.split("\n") if ln.strip()]
    return Template(raw=text, fields=fields, impression=impression, findings_free=free)


# ------------------------------------------------------------------ render


def render_report(
    field_texts: list[tuple[str, str]],
    impression_lines: list[str],
    extra_paragraphs: list[str] | None = None,
    blank_between_fields: bool = True,
    merge_extras: bool = False,
) -> str:
    """Assemble the final report.

    `field_texts` is an ordered list of (UPPERCASE label, body text) pairs; the
    body may be empty for group headers and for OTHER FINDINGS.
    """
    lines: list[str] = ["FINDINGS:"]
    first = True
    for label, body in field_texts:
        if blank_between_fields and not first:
            lines.append("")
        first = False
        body = squash(body)
        if not label:
            if body:
                lines.append(body)
            continue
        lines.append(f"{label}: {body}".rstrip() if body else f"{label}:")
    paras = [squash(p) for p in (extra_paragraphs or []) if squash(p)]
    if merge_extras and paras:
        paras = [" ".join(paras)]
    for para in paras:
        lines.append("")
        lines.append(para)
    lines.append("")
    lines.append("IMPRESSION:")
    lines.extend(squash(x) for x in impression_lines if squash(x))
    out = "\n".join(lines).rstrip() + "\n"
    return out


def resolve_placeholders(text: str, laterality: str | None, region: str | None) -> str:
    """Fill `[left/right]` / `[generic]` style slots left in template prose."""
    if "[" not in text:
        return text

    def repl(m: re.Match) -> str:
        inner = m.group(1).strip()
        low = inner.lower()
        if any(w in low for w in ("left", "right", "bilateral", "laterality")):
            return laterality or ""
        if "generic" in low or "region" in low or "body" in low:
            return region or ""
        if "/" in inner:  # unresolved option list -> first option
            return inner.split("/")[0].strip()
        return ""

    text = re.sub(r"\[([^\]]*)\]", repl, text)
    text = re.sub(r"\s{2,}", " ", text)
    text = re.sub(r"\s+([,.;:])", r"\1", text)
    text = re.sub(r"\bthe\s+(?=[.,])", "", text)
    return cap_first(text.strip())


### 2.4 Dictation segmentation - cues, boilerplate, dictated summary

In [ ]:
%%writefile rrh/dictation.py
"""Dictation segmentation.

Turns a telegraphic dictation into structured `Segment`s, separating
   * technique / history / recommendation boilerplate (never reported),
   * the body of observations, and
   * a trailing radiologist summary (the dictated impression), which the
     reference reports reuse almost verbatim in IMPRESSION.
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field as dc_field

from .lexicon import correct_spelling, normalize_shorthand
from .textutil import clean_ws, content_tokens, jaccard, split_sentences, squash, token_set

CUE_VERB = re.compile(
    r"^(?P<cue>[A-Za-z][A-Za-z0-9 ,/&'\-\.]{1,45}?)\s+(?:shows?|demonstrates?|reveals?|"
    r"demonstrate|reveal)\s+(?P<rest>.+)$",
    re.I,
)
CUE_COLON = re.compile(r"^(?P<cue>[A-Za-z][A-Za-z0-9 ,/&'\-\.]{1,45}?)\s*:\s*(?P<rest>.+)$")
BARE_HEADER = re.compile(r"^(?P<cue>[A-Za-z][A-Za-z0-9 ,/&'\-\.]{1,45}?)\s*:\s*$")
LEVEL_RE = re.compile(r"\b([CTLS])\s*(\d{1,2})\s*[-–/]\s*(?:[CTLS])?\s*(\d{1,2}|S1)\b", re.I)

TECHNIQUE_PAT = re.compile(
    r"(was|were)\s+(performed|obtained|acquired)|"
    r"^(multiplanar|multisequence|multi-planar|axial|sagittal|coronal|sequences?|"
    r"technique|protocol|images?\s+were|imaging\s+was|scout|localizer|"
    r"post-?processed|reformat\w*|reconstruct\w*|magnetic\s+resonance|"
    r"computed\s+tomograph\w*|hrct\b|thin[- ]section)\b|"
    r"\b(radiograph|view|projection)s?\s+(of|were|was|obtained|acquired)\b|"
    r"^\s*(\d+|two|three|four|five|six|single|frontal|lateral|ap and lateral)[\w\- ]{0,25}"
    r"(views?|radiographs?|projections?)\b|"
    r"\b(without|with)\s+(intravenous|iv)\s+contrast\s*$|"
    r"^(contrast|comparison|indication|history|clinical\s+history|technique|exam(ination)?)\s*[:\-]",
    re.I,
)
HISTORY_PAT = re.compile(
    r"\b(pain|swelling|trauma|injury|fall|weakness|numbness|tingling|discomfort|"
    r"complaint|symptoms?)\s+(for|since|x)\s+\d|"
    r"^(clinical\s+)?(history|indication|hx)\b|"
    r"\b(rule\s+out|r/o)\b\s*$|"
    # "15-19-year-old with back pain (M54.9)." - age band, referral, ICD code
    r"\b\d{1,3}\s*(?:-\s*\d{1,3}\s*)?[- ]year[- ]old\b|"
    r"\(\s*[A-TV-Z]\d{2}(?:\.\d+)?\s*\)|"
    r"^(shortness of breath|sob|chest pain|back pain|abdominal pain|headache|fever)\b",
    re.I,
)
RECOMMEND_PAT = re.compile(
    r"^(recommendation|recommend(ed|s)?\b|correlate\s+clinic|"
    r"suggest\s+clinical|further\s+evaluation\s+with|"
    r"clinical\s+correlation\s+(is\s+)?(recommended|suggested|advised))",
    re.I,
)
ADVISED_PAT = re.compile(
    r"\b(is|are)\s+(advised|recommended|suggested|indicated)\b|"
    r"\bfor\s+further\s+(characteri[sz]ation|evaluation|assessment|workup)\b|"
    r"\bcorrelation\s+is\s+(advised|recommended|suggested)\b",
    re.I,
)
NONE_PAT = re.compile(
    r"^(none|n/?a|nil|not applicable)(\s+(available|provided|performed|obtained))?\s*\.?$|"
    r"^("
    r"(no\s+)?(prior|previous|comparison)s?(\s+(study|studies|exam\w*|imaging))?"
    r"(\s+(are|is|were|was))?(\s+(available|provided|performed))?)\s*\.?$",
    re.I,
)
MODALITY_HEADER = re.compile(
    r"^(mri|mr|ct|cta|mra|mrcp|us|usg|ultrasound|sonograph\w*|x-?ray|xr|radiograph\w*|"
    r"pet|dexa|fluoroscop\w*)\b",
    re.I,
)
CONTRAST_PAT = re.compile(
    r"\b(gadavist|gadolinium|gadobutrol|omnipaque|iohexol|ioversol|isovue|optiray|"
    r"ultravist|visipaque|contrast\s+material|contrast\s+agent)\b|"
    r"^\s*\w+\s+\d+(?:\.\d+)?\s*(?:ml|cc|mg)\s+(?:iv|i\.v\.)\b|"
    r"\b\d+(?:\.\d+)?\s*(?:ml|cc)\b[^.]{0,40}\b(administered|injected|intravenous(?:ly)?|"
    r"orally|per\s+oral)\b",
    re.I,
)
SYMPTOM_PAT = re.compile(
    r"\b(nausea|vomiting|headaches?|dizziness|vertigo|fever|chills|cough|dyspn(?:o?ea)|"
    r"seizures?|syncope|palpitations?|fatigue|malaise|tingling|paresthesias?|"
    r"restricted movement|difficulty|inability)\b",
    re.I,
)
ALLCAPS_HEADER = re.compile(r"^[A-Z0-9][A-Z0-9 /\-\(\)\.,&]{6,}$")
NORMAL_PAT = re.compile(
    r"^(normal|norml|nomal|normal study|unremarkable|nad|wnl|within normal limits?|"
    r"no abnormality|no acute abnormality|essentially normal|grossly normal)\s*\.?$",
    re.I,
)

GLOBAL_CONCL = re.compile(
    r"^(no acute|no significant|otherwise\s+(unremarkable|normal)|overall|"
    r"essentially\s+normal|unremarkable\s+(study|examination))\b.*"
    r"(abnormality|abnormalities|finding|study|examination|"
    r"\b(shoulder|knee|hip|wrist|ankle|elbow|hand|foot|spine|chest|abdomen|pelvis|"
    r"brain|head|neck|femur|humerus|skull|thorax)\b)",
    re.I,
)

NEG_LEAD = re.compile(
    r"^(no\b|not\b|without\b|absence of\b|negative for\b|there is no\b|there are no\b|"
    r"free of\b|unremarkable\b|normal\b|intact\b|preserved\b|maintained\b)",
    re.I,
)
NEG_ANY = re.compile(r"\b(no|not|without|negative for|absence of|free of)\b", re.I)
POS_HINT = re.compile(
    r"\b(mild|moderate|severe|small|large|minimal|marked|mild-to-moderate|"
    r"moderate-to-severe|trace|prominent|focal|diffuse|multiple|few|grade\s*[iv1-4]|"
    r"partial|complete|acute|chronic|degenerative|tear|fracture|effusion|edema|oedema|"
    r"stenosis|narrowing|osteophyte|bulge|herniation|opacity|consolidation|nodule|mass|"
    r"lesion|cyst|swelling|hypertrophy|spondylosis|tendinosis|bursitis|arthrosis|"
    r"arthritis|atrophy|thickening|calcification|sclerosis|deformity|dislocation|"
    r"collection|abnormal)\b",
    re.I,
)
LATERAL_RE = re.compile(r"\b(right|left|bilateral|rt|lt|b/l|bilat)\b", re.I)
MEASURE_RE = re.compile(r"\b\d+(?:\.\d+)?\s*(?:x\s*\d+(?:\.\d+)?\s*)*(?:mm|cm|ml|cc)\b", re.I)


@dataclass
class Segment:
    text: str
    cue: str | None = None
    own_cue: bool = False  # the cue was written in this sentence, not inherited
    kind: str = "finding"  # finding | preamble | impression | normal
    negative: bool = False
    tokens: frozenset = dc_field(default_factory=frozenset)

    def __post_init__(self) -> None:
        if not self.tokens:
            self.tokens = token_set(self.text)
        self.negative = bool(NEG_LEAD.match(self.text)) and not POS_HINT.search(self.text)

    @property
    def laterality(self) -> str | None:
        m = LATERAL_RE.search(self.text)
        if not m:
            return None
        v = m.group(1).lower()
        return {"rt": "right", "lt": "left", "b/l": "bilateral", "bilat": "bilateral"}.get(v, v)


@dataclass
class DictationDoc:
    raw: str
    preamble: list[Segment]
    findings: list[Segment]
    impression: list[Segment]
    is_normal: bool

    @property
    def all_reportable(self) -> list[Segment]:
        return self.findings + self.impression


def _classify_boilerplate(sent: str) -> str | None:
    s = sent.strip()
    if NONE_PAT.match(s):
        return "preamble"
    if RECOMMEND_PAT.match(s) or ADVISED_PAT.search(s):
        return "preamble"
    if HISTORY_PAT.search(s):
        return "preamble"
    if TECHNIQUE_PAT.search(s):
        return "preamble"
    if ALLCAPS_HEADER.match(s) and not LEVEL_RE.search(s):
        return "preamble"
    if MODALITY_HEADER.match(s) and not POS_HINT.search(s) and not NEG_ANY.search(s):
        return "preamble"
    if CONTRAST_PAT.search(s) and len(content_tokens(s)) <= 8:
        return "preamble"
    if SYMPTOM_PAT.search(s) and len(content_tokens(s)) <= 6:
        return "preamble"
    return None


ARTICLE_LEAD = re.compile(r"^(the|a|an|there|this|these|those|it|his|her|their)\b", re.I)


def _strip_cue(sent: str) -> tuple[str | None, str, bool]:
    """Return (routing cue, sentence body).

    A *header* cue ("Bones shows ...", "Labrum: ...") is removed from the text,
    exactly as the reference reports do.  A cue that is really the subject of a
    normal sentence ("The medial meniscus demonstrates ...") is kept verbatim
    and only used for routing.
    """
    m = CUE_VERB.match(sent)
    if m:
        cue, rest = m.group("cue").strip(), m.group("rest").strip()
        if len(content_tokens(cue)) <= 6 and rest:
            if ARTICLE_LEAD.match(cue) or rest[:1].isupper():
                return cue, sent, False
            return cue, rest, True
    m = CUE_COLON.match(sent)
    if m:
        cue, rest = m.group("cue").strip(), m.group("rest").strip()
        if len(cue.split()) <= 6 and not re.search(r"[.!?]", cue) and rest:
            return cue, rest, True
    return None, sent, False


def _match_score(a: Segment, prior: list[Segment]) -> float:
    best = 0.0
    for p in prior:
        best = max(best, jaccard(a.tokens, p.tokens))
        if best >= 0.9:
            break
    return best


def _detect_summary(segs: list[Segment], threshold: float = 0.34, max_misses: int = 3,
                    after_cues: bool = False) -> int:
    """Index where the dictated impression starts (len(segs) if there is none).

    Two independent signals: a global conclusion sentence in the back part of
    the dictation, and a trailing run of sentences that restate earlier ones.
    """
    n = len(segs)
    if n < 4:
        return n
    body_limit = max(2, n // 3)
    # Section cues ("C5-6 shows ...", "Bones show ...") mark the body of the
    # dictation: a summary can only start after the last one.  Without this,
    # level-by-level spine dictations look like a repeated-sentence summary.
    last_cue = max((i for i, s in enumerate(segs) if s.own_cue), default=-1)
    body_limit = max(body_limit, last_cue + 1)
    if body_limit >= n - 1:
        return n
    if after_cues and last_cue >= 0 and n - (last_cue + 1) >= 2:
        # a structured dictation ends its cued sections and then summarises
        return last_cue + 1

    marker = n
    for i in range(body_limit, n - 1):
        if GLOBAL_CONCL.match(segs[i].text):
            marker = i
            break

    dup = n
    misses = 0
    for i in range(n - 1, body_limit - 1, -1):
        if _match_score(segs[i], segs[:i]) >= threshold:
            dup = i
            misses = 0
        else:
            misses += 1
            if misses > max_misses:
                break
    if dup >= n - 1:
        dup = n

    start = min(marker, dup)
    if start >= n - 1:
        return n
    while start > body_limit and GLOBAL_CONCL.match(segs[start - 1].text):
        start -= 1
    return start


def segment_dictation(
    raw: str,
    normalize: bool = True,
    vocab: dict[str, int] | None = None,
    summary_threshold: float = 0.34,
    summary_max_misses: int = 3,
    summary_after_cues: bool = False,
) -> DictationDoc:
    text = clean_ws(raw or "")
    if normalize:
        text = normalize_shorthand(text)
    if vocab:
        text = correct_spelling(text, vocab)
    if not text or NORMAL_PAT.match(squash(text)):
        return DictationDoc(raw=text, preamble=[], findings=[], impression=[], is_normal=True)

    sents = split_sentences(text)
    pre: list[Segment] = []
    body: list[Segment] = []
    carry_cue: str | None = None
    for s in sents:
        s = squash(s)
        if not s:
            continue
        kind = _classify_boilerplate(s)
        if kind == "preamble":
            pre.append(Segment(text=s, kind="preamble"))
            continue
        if NORMAL_PAT.match(s):
            continue
        bare = BARE_HEADER.match(s)
        if bare:  # a section header on its own line: routing cue, not a finding
            carry_cue = bare.group("cue").strip()
            continue
        cue, rest, is_header = _strip_cue(s)
        _own = cue
        if is_header:
            carry_cue = cue
        elif carry_cue and not cue:
            cue = carry_cue
        body.append(Segment(text=squash(rest), cue=cue, own_cue=bool(cue) and cue == _own))
    # a cue must not leak into the dictated summary


    body = [b for b in body if content_tokens(b.text)]
    cut = _detect_summary(body, summary_threshold, summary_max_misses, summary_after_cues)
    findings, impression = body[:cut], body[cut:]
    for seg in impression:
        seg.kind = "impression"
    is_normal = not body
    return DictationDoc(
        raw=text, preamble=pre, findings=findings, impression=impression, is_normal=is_normal
    )


### 2.5 Coordinated-clause splitting

In [ ]:
%%writefile rrh/splitting.py
"""Decompose coordinated dictation sentences so each clause can be routed.

The reference reports routinely split a dictated sentence across fields, e.g.

    "No acute fracture or dislocation is identified."
        -> BONES:  No acute fracture is identified.
        -> JOINTS: No dislocation is identified.

A decomposition is only adopted (in `routing`) when its parts genuinely land in
different template fields; otherwise the sentence is kept verbatim.
"""
from __future__ import annotations

import re

from .textutil import content_tokens, squash

_NEG_LEAD = re.compile(
    r"^(?P<lead>no evidence of|there is no|there are no|no significant|no acute|no)\s+"
    r"(?P<body>.+?)(?P<tail>\s+(?:is|are|was|were)\s+"
    r"(?:identified|seen|noted|present|evident|visualized|visualised|appreciated|"
    r"demonstrated|observed))?\s*[.]?$",
    re.I,
)
_SUBJ_LIST = re.compile(
    r"^(?P<det>the\s+)?(?P<subj>.+?)\s+(?P<verb>are|were)\s+(?P<pred>.+?)\s*[.]?$", re.I
)
_LIST_SPLIT = re.compile(r"\s*,\s*(?:and\s+|or\s+)?|\s+and\s+|\s+or\s+", re.I)
_COMMA_SPLIT = re.compile(r"\s*[,;]\s*")

_MAX_PART_TOKENS = 8


def _ok_parts(parts: list[str], minimum: int = 2) -> bool:
    if len(parts) < minimum:
        return False
    for p in parts:
        n = len(content_tokens(p))
        if n < 1 or n > _MAX_PART_TOKENS:
            return False
    return True


def split_negation(text: str) -> list[str] | None:
    m = _NEG_LEAD.match(squash(text))
    if not m:
        return None
    lead = m.group("lead")
    body = m.group("body") or ""
    tail = m.group("tail") or ""
    if re.search(r"\b(with|without|causing|producing|resulting|extending)\b", body, re.I):
        return None
    parts = [p.strip() for p in _LIST_SPLIT.split(body) if p.strip()]
    if not _ok_parts(parts):
        return None
    tail = re.sub(r"\bare\b", "is", tail, flags=re.I)
    tail = re.sub(r"\bwere\b", "was", tail, flags=re.I)
    low = lead.lower()
    if low == "no evidence of":
        return [f"No evidence of {p}{tail}." for p in parts]
    if low in {"no acute", "no significant"}:
        # the qualifier belongs to the first item only:
        # "No acute fracture or dislocation" -> "No acute fracture." / "No dislocation."
        head = f"{lead.capitalize().replace('No ', 'No ')} {parts[0]}{tail}."
        return [head] + [f"No {p}{tail}." for p in parts[1:]]
    return [f"No {p}{tail}." for p in parts]


def split_subject_list(text: str) -> list[str] | None:
    t = squash(text)
    m = _SUBJ_LIST.match(t)
    if not m:
        return None
    subj, pred = m.group("subj"), m.group("pred")
    det = "The " if (m.group("det") or "").strip() else ""
    if re.search(r"\b(no|not|without)\b", subj, re.I):
        return None
    parts = [p.strip() for p in _LIST_SPLIT.split(subj) if p.strip()]
    if not _ok_parts(parts):
        return None
    verb = "is" if m.group("verb").lower() == "are" else "was"
    return [f"{det}{p} {verb} {pred}." for p in parts]


def split_clauses(text: str) -> list[str] | None:
    t = squash(text).rstrip(".")
    parts = [p.strip(" .") for p in _COMMA_SPLIT.split(t) if p.strip(" .")]
    if not _ok_parts(parts):
        return None
    if any(re.match(r"^(and|or|with|without|which|more|greatest|most)\b", p, re.I) for p in parts):
        return None
    return [p + "." for p in parts]


def candidate_splits(text: str) -> list[list[str]]:
    """Alternative decompositions of `text`, best first."""
    out: list[list[str]] = []
    for fn in (split_negation, split_subject_list, split_clauses):
        parts = fn(text)
        if parts and len(parts) >= 2:
            out.append([squash(p) for p in parts])
    # de-duplicate while preserving order
    seen: set[tuple[str, ...]] = set()
    uniq: list[list[str]] = []
    for p in out:
        k = tuple(p)
        if k not in seen:
            seen.add(k)
            uniq.append(p)
    return uniq


### 2.6 Field routing (mined from the training reports)

In [ ]:
%%writefile rrh/routing.py
"""Field routing: decide which template field each dictated finding belongs to.

Signals, in decreasing order of trust:
  1. an explicit dictation cue ("Bones shows ...", "L4-L5: ...")
  2. the field label itself appearing in the finding
  3. curated anatomy->field concepts (`lexicon.py`)
  4. overlap with the field's own normal statement in the template
  5. statistics mined from the training reports

The mined statistics are fitted only on training rows, so the evaluation can
hold folds out honestly.
"""
from __future__ import annotations

import math
import re
from collections import Counter, defaultdict
from dataclasses import dataclass, field as dc_field

from .lexicon import concept_for_label, concept_hits, stem
from .template import Template, parse_template
from .textutil import content_tokens, key, sim, split_sentences, squash

LEVEL_RE = re.compile(r"^([ctls])\s*(\d{1,2})\s*[-/]\s*(?:([ctls])\s*)?(\d{1,2}|s1)$", re.I)

# Reporting verbs and connectives a reference sentence may add without adding
# any clinical content.
NEUTRAL_STEMS = {
    stem(w)
    for w in (
        "present", "noted", "seen", "identified", "evident", "demonstrated", "observed",
        "visualized", "visualised", "appreciated", "there", "remaining", "otherwise",
        "again", "also", "overall", "appears", "appear", "the", "are", "is", "which",
    )
}


def normalize_level(label: str) -> str | None:
    lab = key(label).replace(" ", "")
    lab = re.sub(r"^([ctls])(\d{1,2})([ctls])?(\d{1,2})$", r"\1\2-\4", lab)
    m = LEVEL_RE.match(lab.replace(" ", ""))
    if not m:
        m = LEVEL_RE.match(re.sub(r"([a-z])(\d+)-([a-z]?)(\d+)", r"\1\2-\4", lab))
    if not m:
        return None
    a, n1, _, n2 = m.groups()
    return f"{a.lower()}{int(n1)}-{n2.lower() if not n2.isdigit() else int(n2)}"


def label_tokens(label: str) -> set[str]:
    return {stem(t) for t in content_tokens(label)}


def _label_match(text: str, label: str, weights: dict[str, float] | None = None) -> float:
    """How strongly does `text` name `label`?"""
    if not label:
        return 0.0
    lv, tv = normalize_level(label), normalize_level(text)
    if lv and tv:
        return 1.0 if lv == tv else 0.0
    if lv and not tv:
        toks = re.findall(r"\b[ctls]\s*\d{1,2}\s*[-/]\s*[ctls]?\s*(?:\d{1,2}|s1)\b", key(text))
        if any(normalize_level(t) == lv for t in toks):
            return 1.0
        return 0.0
    lt = label_tokens(label)
    if not lt:
        return 0.0
    tt = {stem(t) for t in content_tokens(text)}
    if not tt:
        return 0.0
    if weights:
        num = sum(weights.get(t, 1.0) for t in sorted(lt & tt))
        den = sum(weights.get(t, 1.0) for t in sorted(lt)) or 1.0
        return min(1.0, num / den)
    return len(lt & tt) / len(lt)


@dataclass
class RoutingModel:
    token_label: dict[str, Counter] = dc_field(default_factory=lambda: defaultdict(Counter))
    label_total: Counter = dc_field(default_factory=Counter)
    token_total: Counter = dc_field(default_factory=Counter)
    n_pairs: int = 0

    postings: dict[str, list[int]] = dc_field(default_factory=lambda: defaultdict(list))
    examples: list[tuple[frozenset, str]] = dc_field(default_factory=list)
    vocab: dict[str, int] = dc_field(default_factory=dict)
    ranker: object | None = None
    field_edits: dict[tuple[str, str], tuple[int, int]] = dc_field(default_factory=dict)
    label_edits: dict[str, tuple[int, int]] = dc_field(default_factory=dict)
    global_edit_rate: float = 0.35

    def edit_prior(self, tkey: str, label: str, alpha: float = 2.0) -> float:
        """How often the reference edits this field of this template."""
        lab_e, lab_n = self.label_edits.get(label, (0, 0))
        backoff = (lab_e + alpha * self.global_edit_rate) / (lab_n + alpha)
        e, n = self.field_edits.get((tkey, label), (0, 0))
        return (e + alpha * backoff) / (n + alpha)
    style_examples: list[tuple[frozenset, str, str]] = dc_field(default_factory=list)
    style_postings: dict[str, list[int]] = dc_field(default_factory=lambda: defaultdict(list))

    def style_match(self, clause: str, threshold: float) -> str | None:
        """Closest training clause->reported-sentence rewrite, if it is safe.

        "Safe" means the reference sentence introduces no content word that the
        clause does not already have (apart from neutral reporting verbs), so
        this transfers phrasing only - never a finding.
        """
        toks = {stem(t) for t in content_tokens(clause)}
        if not toks or not self.style_examples:
            return None
        cand: Counter = Counter()
        for t in sorted(toks):
            for i in self.style_postings.get(t, ()):  # type: ignore[arg-type]
                cand[i] += 1
        best, best_s = None, 0.0
        for i, _ in sorted(cand.items(), key=lambda kv: (-kv[1], kv[0]))[:60]:
            etoks, src, tgt = self.style_examples[i]
            j = len(toks & etoks) / len(toks | etoks)
            if j < threshold:
                continue
            score = j + 0.001 * len(etoks)
            if score > best_s:
                best, best_s = (src, tgt), score
        if not best:
            return None
        _, target = best
        extra = {stem(t) for t in content_tokens(target)} - toks - NEUTRAL_STEMS
        if extra:
            return None
        return target

    def knn(self, tokens: set[str], allowed: set[str], k: int = 12) -> dict[str, float]:
        """Similarity-weighted vote of the closest mined training sentences."""
        if not tokens or not self.examples:
            return {}
        cand: Counter = Counter()
        for t in sorted(tokens):
            for i in self.postings.get(t, ()):  # type: ignore[arg-type]
                cand[i] += 1
        if not cand:
            return {}
        scored = []
        for i, _ in sorted(cand.items(), key=lambda kv: (-kv[1], kv[0]))[:120]:
            etoks, label = self.examples[i]
            if label not in allowed:
                continue
            inter = len(tokens & etoks)
            if not inter:
                continue
            j = inter / len(tokens | etoks)
            scored.append((j, label))
        scored.sort(reverse=True)
        votes: dict[str, float] = {}
        tot = 0.0
        for j, label in sorted(scored[:k]):
            votes[label] = votes.get(label, 0.0) + j
            tot += j
        if tot <= 0:
            return {}
        return {lab: v / tot for lab, v in votes.items()}

    def score(self, tokens: set[str], label: str) -> float:
        if label not in self.label_total or not tokens:
            return 0.0
        total = self.label_total[label]
        acc = 0.0
        for t in sorted(tokens):
            c = self.token_label.get(t)
            if not c:
                continue
            p_t_l = c.get(label, 0) / total
            p_t = self.token_total[t] / max(1, self.n_pairs)
            if p_t_l > 0:
                acc += math.log((p_t_l + 1e-4) / (p_t + 1e-4))
        return max(0.0, min(1.0, acc / (2.0 * max(1, len(tokens)))))


def mine_row(row) -> list[tuple[int, str, str, str]]:
    """(segment index, segment text, gold field label, reference sentence)."""
    from .dictation import segment_dictation

    tmpl = parse_template(row["template_content"])
    rep = parse_template(row["report"])
    doc = segment_dictation(row["dictation"])
    units = doc.findings or doc.impression
    if not units:
        return []
    tmpl_by_label = {f.label: f for f in tmpl.fields}
    out: list[tuple[int, str, str]] = []
    used: set[int] = set()
    for rf in rep.fields:
        if not rf.label:
            continue
        tf = tmpl_by_label.get(rf.label)
        tmpl_sents = tf.sentences if tf else []
        for rs in split_sentences(rf.text):
            if any(sim(rs, ts) >= 0.85 for ts in tmpl_sents):
                continue
            best, best_s = -1, 0.0
            for i, u in enumerate(units):
                s = sim(rs, u.text)
                if s > best_s:
                    best, best_s = i, s
            if best >= 0 and best_s >= 0.5 and best not in used:
                used.add(best)
                out.append((best, units[best].text, rf.label, rs))
    return out


def mine_pairs(rows) -> list[tuple[str, str]]:
    """(dictated sentence, report field label) supervision mined from train rows."""
    return [(text, label) for row in rows for _, text, label, _ in mine_row(row)]


def fit_router(rows, cfg=None) -> RoutingModel:
    from .lexicon import build_vocabulary

    model = RoutingModel()
    model.vocab = build_vocabulary(
        [r.get("report") or "" for r in rows] + [r.get("template_content") or "" for r in rows]
    )
    edited: dict[tuple[str, str], list[int]] = {}
    lab_edit: dict[str, list[int]] = {}
    tot_e = tot_n = 0
    for row in rows:
        tkey = template_key(row.get("template_content") or "")
        tmpl_fields = {f.label: squash(f.text) for f in parse_template(row["template_content"]).fields if f.label}
        rep_fields = {f.label: squash(f.text) for f in parse_template(row["report"]).fields if f.label}
        for lab, txt in tmpl_fields.items():
            changed = int(rep_fields.get(lab, "") != txt)
            edited.setdefault((tkey, lab), [0, 0])
            edited[(tkey, lab)][0] += changed
            edited[(tkey, lab)][1] += 1
            lab_edit.setdefault(lab, [0, 0])
            lab_edit[lab][0] += changed
            lab_edit[lab][1] += 1
            tot_e += changed
            tot_n += 1
    model.field_edits = {k: (v[0], v[1]) for k, v in edited.items()}
    model.label_edits = {k: (v[0], v[1]) for k, v in lab_edit.items()}
    model.global_edit_rate = tot_e / max(1, tot_n)

    pairs: list[tuple[str, str]] = []
    for row in rows:
        for _, text, label, ref_sentence in mine_row(row):
            pairs.append((text, label))
            toks = frozenset(stem(t) for t in content_tokens(text))
            if not toks:
                continue
            idx = len(model.style_examples)
            model.style_examples.append((toks, text, ref_sentence))
            for t in toks:
                model.style_postings[t].append(idx)
    for text, label in pairs:
        toks = {stem(t) for t in content_tokens(text)}
        if not toks:
            continue
        model.n_pairs += 1
        model.label_total[label] += 1
        idx = len(model.examples)
        model.examples.append((frozenset(toks), label))
        for t in toks:
            model.token_label[t][label] += 1
            model.token_total[t] += 1
            model.postings[t].append(idx)
    return model


WEIGHTS = {
    "cue": 3.0,
    "label": 2.2,
    "concept": 1.2,
    "template": 1.0,
    "mined": 1.2,
    "knn": 1.6,
    "continuity": 0.0,
    "backward": 0.0,
    "prior": 0.0,
}
MIN_SCORE = 0.30


def field_candidates(tmpl: Template):
    return [f for f in tmpl.fields if f.label and not f.is_free and f.label != "OTHER FINDINGS"]


def cue_supported(cue: str | None, tmpl: Template, ctx: "TemplateContext",
                  minimum: float = 0.34) -> bool:
    """Does any field of this template correspond to the dictation's cue?

    When the radiologist dictates "Brain shows ..." but the template has no
    brain field, the finding belongs to no field at all - the reference reports
    put it in a trailing paragraph rather than forcing it into a wrong field.
    """
    if not cue:
        return True
    hits = concept_hits(cue)
    for f in field_candidates(tmpl):
        if _label_match(cue, f.label, ctx.label_weights) >= minimum:
            return True
        concept = concept_for_label(f.label)
        if concept and concept in hits:
            return True
    return False


FEATURES = (
    "cue", "label", "concept", "template", "mined", "knn",
    "group", "same_prev", "backward", "forward", "prior",
)


def template_key(text: str) -> str:
    import hashlib

    return hashlib.sha256(key(text).encode()).hexdigest()[:16]


def field_features(seg_text: str, cue: str | None, tmpl: Template, model: RoutingModel,
                   ctx: "TemplateContext",
                   prev_order: int | None = None) -> list[tuple[str, dict[str, float]]]:
    """Per-candidate-field feature vector for one dictated clause.

    The same features drive the hand-weighted scorer and the learned ranker, so
    the two are directly comparable.
    """
    idf, lw = ctx.idf, ctx.label_weights
    toks = {stem(t) for t in content_tokens(seg_text)}
    hits = concept_hits(seg_text)
    tot_hits = sum(hits.values())
    cands = field_candidates(tmpl)
    knn = model.knn(toks, {f.label for f in cands})
    span = max(1, len(tmpl.fields) - 1)
    out: list[tuple[str, dict[str, float]]] = []
    for f in cands:
        feat = {k: 0.0 for k in FEATURES}
        if cue:
            feat["cue"] = _label_match(cue, f.label, lw)
        feat["label"] = _label_match(seg_text, f.label, lw)
        concept = concept_for_label(f.label)
        if concept and tot_hits:
            feat["concept"] = hits.get(concept, 0) / tot_hits
        if f.text:
            ft = {stem(t) for t in content_tokens(f.text)}
            if ft and toks:
                num = sum(idf.get(t, 1.0) for t in sorted(ft & toks))
                den = sum(idf.get(t, 1.0) for t in sorted(ft)) or 1.0
                feat["template"] = min(1.0, num / den)
        feat["mined"] = model.score(toks, f.label)
        feat["knn"] = knn.get(f.label, 0.0)
        feat["group"] = 1.0 if f.is_group else 0.0
        feat["prior"] = model.edit_prior(ctx.template_key, f.label)
        if prev_order is not None:
            if f.order == prev_order:
                feat["same_prev"] = 1.0
            elif f.order < prev_order:
                feat["backward"] = min(1.0, (prev_order - f.order) / span)
            else:
                feat["forward"] = min(1.0, (f.order - prev_order) / span)
        out.append((f.label, feat))
    return out


def score_segment(seg_text: str, cue: str | None, tmpl: Template, model: RoutingModel,
                  ctx: "TemplateContext",
                  weights: dict[str, float] | None = None,
                  prev_order: int | None = None) -> list[tuple[float, str]]:
    """Score every candidate field for one dictated clause.

    `prev_order` is the template position of the field the previous clause went
    to.  Radiologists dictate in template order - 52% of consecutive findings
    stay in the same field and 85% never move backwards - so continuity is a
    real signal, not a heuristic.
    """
    W = weights or WEIGHTS
    scored: list[tuple[float, str]] = []
    for label, feat in field_features(seg_text, cue, tmpl, model, ctx, prev_order):
        if model.ranker is not None and W.get("use_ranker"):
            s = model.ranker.score(feat)
        else:
            s = (
                W["cue"] * feat["cue"]
                + W["label"] * feat["label"]
                + W["concept"] * feat["concept"]
                + W["template"] * feat["template"]
                + W["mined"] * feat["mined"]
                + W["knn"] * feat["knn"]
                + W.get("prior", 0.0) * feat["prior"]
                + W.get("continuity", 0.0) * feat["same_prev"]
                - W.get("backward", 0.0) * feat["backward"]
            )
        scored.append((s, label))
    scored.sort(key=lambda x: (-x[0], x[1]))
    return scored


@dataclass
class TemplateContext:
    idf: dict[str, float]
    label_weights: dict[str, float]
    template_key: str = ""


def build_context(tmpl: Template) -> TemplateContext:
    """Rarity weights computed over the template's own fields and labels:
    a word that occurs in only one field/label is decisive for that field."""
    fields = field_candidates(tmpl)
    n = max(1, len(fields))
    df: Counter = Counter()
    for f in fields:
        for t in {stem(x) for x in content_tokens(f.text)}:
            df[t] += 1
    idf = {t: math.log((n + 1) / (c + 0.5)) for t, c in df.items()}

    ldf: Counter = Counter()
    for f in fields:
        for t in label_tokens(f.label):
            ldf[t] += 1
    lw = {t: math.log((n + 1) / (c + 0.5)) for t, c in ldf.items()}
    # generic label words carry little routing information
    for generic in ("structure", "space", "tissu", "other", "finding", "region", "gener"):
        for t in list(lw):
            if t.startswith(generic):
                lw[t] = min(lw[t], 0.3)
    return TemplateContext(idf=idf, label_weights=lw, template_key=template_key(tmpl.raw))


# backwards-compatible helper
def build_idf(tmpl: Template) -> TemplateContext:
    return build_context(tmpl)


def fit_ranked_router(rows, cfg) -> RoutingModel:
    """Fit the mined statistics, then fit the learned router on top of them."""
    from .ranker import build_examples, fit_ranker

    model = fit_router(rows)
    if getattr(cfg, "use_ranker", False):
        model.ranker = fit_ranker(
            build_examples(rows, model, cfg),
            epochs=cfg.ranker_epochs,
            lr=cfg.ranker_lr,
            l2=cfg.ranker_l2,
        )
    return model


### 2.7 Template editing - replace only what is contradicted

In [ ]:
%%writefile rrh/editor.py
"""Template editing: fold routed findings into the template's normal statements.

Guiding rule (RULE 2/3/6 of the task): replace only the normal statement that
the dictation contradicts, keep everything else byte-identical to the template.
"""
from __future__ import annotations

import re

from .lexicon import stem
from .splitting import split_negation
from .textutil import content_tokens, sim, split_sentences, squash, tidy_sentence

NEGATIVE_RE = re.compile(r"^\s*(no|there is no|there are no|without|negative for)\b", re.I)
NORMAL_STATE_RE = re.compile(
    r"\b(unremarkable|normal|intact|preserved|maintained|within normal limits|clear|"
    r"patent|not widened|no significant)\b",
    re.I,
)
BLANKET_RE = re.compile(
    r"^(the\s+)?[\w\s/-]{0,40}\s*(is|are)\s+(unremarkable|normal|preserved|maintained|"
    r"intact|clear|within normal (size )?limits?)\.?$",
    re.I,
)
FINITE_VERB = re.compile(
    r"\b(is|are|was|were|has|have|had|shows?|demonstrates?|reveals?|appears?|measures?|"
    r"noted|seen|identified|present|extends?|involves?|causes?|produces?|results?|"
    r"remains?|persists?|distends?|projects?)\b",
    re.I,
)
PLURAL_TAIL = re.compile(r"(?<![aiou])s$|(?<=[^s])es$", re.I)
PREPOSITION = re.compile(
    r"\b(of|in|at|along|about|within|with|involving|throughout|around|over|between)\b", re.I
)
LEADING_THERE = re.compile(r"^there\b", re.I)

COPULA_TAIL = re.compile(
    r"\s+(?:is|are|was|were)\s+(?:identified|seen|noted|present|evident|visualized|"
    r"visualised|appreciated|demonstrated|observed)\b",
    re.I,
)


def is_negative(sentence: str) -> bool:
    return bool(NEGATIVE_RE.match(sentence.strip()))


def negated_entities(sentence: str) -> list[str]:
    parts = split_negation(sentence)
    if not parts:
        m = re.match(r"^\s*(?:no|there is no|there are no)\s+(.+?)\s*[.]?$", sentence, re.I)
        return [m.group(1)] if m else []
    out = []
    for p in parts:
        m = re.match(r"^\s*No\s+(.+?)\s*[.]?$", p, re.I)
        if m:
            out.append(COPULA_TAIL.sub("", m.group(1)).strip())
    return out


def _stems(text: str) -> set[str]:
    return {stem(t) for t in content_tokens(text)}


def _asserted_positively(entity: str, routed: list[str]) -> bool:
    """Does any routed finding assert `entity` as present?"""
    ent = _stems(entity)
    if not ent:
        return False
    for r in routed:
        rs = _stems(r)
        if not rs:
            continue
        if len(ent & rs) / len(ent) < 0.7:
            continue
        if is_negative(r):
            # the routed sentence also negates it -> not a contradiction
            r_ents = negated_entities(r)
            if any(len(ent & _stems(e)) / len(ent) >= 0.7 for e in r_ents):
                continue
        return True
    return False


def _covered(sentence: str, routed: list[str], threshold: float) -> bool:
    st = _stems(sentence)
    if not st:
        return True
    union: set[str] = set()
    for r in routed:
        union |= _stems(r)
        if sim(sentence, r) >= 0.62:
            return True
    return len(st & union) / len(st) >= threshold


def _rebuild_negative(original: str, kept: list[str]) -> str:
    tail_m = COPULA_TAIL.search(original)
    tail = tail_m.group(0) if tail_m else ""
    if len(kept) == 1:
        body = kept[0]
    else:
        body = ", ".join(kept[:-1]) + " or " + kept[-1]
    return tidy_sentence(f"No {body}{tail}")


def _is_plural_head(text: str) -> bool:
    head = PREPOSITION.split(text, maxsplit=1)[0]
    if re.search(r"\band\b", head, re.I):
        return True
    for w in re.findall(r"[A-Za-z]+", head):
        low = w.lower()
        if low.endswith(("sis", "ss", "us", "is", "ous")):
            continue
        if PLURAL_TAIL.search(low):
            return True
    return False


def render_clause(text: str, cfg) -> str:
    """Render a routed dictation clause as a report sentence.

    Telegraphic noun phrases ("degenerative changes in left shoulder") are given
    the existential frame the reference reports use ("There are degenerative
    changes in the left shoulder.").  No content is added.
    """
    t = squash(text)
    verbless = t and not FINITE_VERB.search(t) and not NEGATIVE_RE.match(t)
    if verbless and not LEADING_THERE.match(t):
        if cfg.add_existential:
            t = f"There {'are' if _is_plural_head(t) else 'is'} {t[0].lower() + t[1:]}"
        elif cfg.add_copula:
            t = f"{t.rstrip('.')} {'are' if _is_plural_head(t) else 'is'} present"
    return tidy_sentence(t)


def _redundant_with_template(clause: str, tmpl_sents: list[str]) -> bool:
    """True when the template already says this, in its own words (RULE 6)."""
    c_ents = negated_entities(clause)
    c_st = _stems(clause)
    if not c_st:
        return True
    for ts in tmpl_sents:
        if sim(clause, ts) >= 0.55:
            return True
        if not is_negative(ts):
            continue
        t_ents = negated_entities(ts)
        if not c_ents or not t_ents:
            continue
        t_all = set()
        for e in t_ents:
            t_all |= _stems(e)
        if all(
            _stems(e) and len(_stems(e) & t_all) / len(_stems(e)) >= 0.7 for e in c_ents
        ):
            return True
    return False


def filter_redundant(routed: list[str], template_text: str, cfg) -> list[str]:
    """Drop dictated clauses that merely restate a template normal statement."""
    if not cfg.suppress_redundant_negatives or not routed:
        return routed
    tmpl_sents = split_sentences(template_text)
    if not tmpl_sents:
        return routed
    out = []
    for r in routed:
        negative = is_negative(r)
        normalish = bool(NORMAL_STATE_RE.search(r))
        if (negative or (cfg.suppress_redundant_normals and normalish)) and (
            _redundant_with_template(r, tmpl_sents)
        ):
            continue
        out.append(r)
    return out


MEASURE_RE = re.compile(r"\b\d+(?:\.\d+)?\s*(?:x\s*\d+(?:\.\d+)?\s*)*(?:mm|cm|ml|cc)\b", re.I)


def dedupe_clauses(routed: list[str], threshold: float) -> list[str]:
    """Drop a clause the field already states (dictated per-level summaries).

    A clause carrying a measurement the field does not have yet is always kept:
    de-duplication must never silently lose a number.
    """
    kept: list[str] = []
    for r in routed:
        st = _stems(r)
        if st:
            prev_text = " ".join(kept)
            prev = _stems(prev_text)
            new_measures = {
                m.group(0).lower().replace(" ", "") for m in MEASURE_RE.finditer(r)
            } - {m.group(0).lower().replace(" ", "") for m in MEASURE_RE.finditer(prev_text)}
            if not new_measures and (
                len(st & prev) / len(st) >= threshold or any(sim(r, k) >= 0.7 for k in kept)
            ):
                continue
        kept.append(r)
    return kept


def edit_field(template_text: str, routed: list[str], cfg) -> str:
    """Return the new body for one field."""
    if cfg.dedupe_threshold:
        routed = dedupe_clauses(routed, cfg.dedupe_threshold)
    routed = filter_redundant(routed, template_text, cfg)
    if not routed:
        return template_text
    kept: list[str] = []
    for ts in split_sentences(template_text):
        if is_negative(ts):
            ents = negated_entities(ts)
            if ents:
                survivors = [e for e in ents if not _asserted_positively(e, routed)]
                if not survivors:
                    continue
                if len(survivors) < len(ents):
                    kept.append(_rebuild_negative(ts, survivors))
                    continue
            if _covered(ts, routed, cfg.cover_threshold):
                continue
            kept.append(ts)
        else:
            if _covered(ts, routed, cfg.cover_threshold):
                continue
            if cfg.soften_blanket and BLANKET_RE.match(ts) and any(
                not is_negative(r) for r in routed
            ):
                kept.append(_soften(ts))
            else:
                kept.append(ts)
    ordered = routed
    if cfg.abnormal_first:
        pos = [r for r in routed if not is_negative(r) and not NORMAL_STATE_RE.search(r)]
        rest = [r for r in routed if r not in pos]
        ordered = pos + rest
    body = " ".join(render_clause(r, cfg) for r in ordered if squash(r))
    if kept:
        body = (body + " " + " ".join(tidy_sentence(k) for k in kept)).strip()
    return squash(body)


def _soften(sentence: str) -> str:
    s = sentence
    s = re.sub(r"^The\s+", "The remaining ", s, count=1)
    s = re.sub(r"\s+(is|are)\s+", lambda m: f" {m.group(1)} otherwise ", s, count=1)
    return tidy_sentence(s)


### 2.8 Impression builder

In [ ]:
%%writefile rrh/impression.py
"""IMPRESSION construction.

Two branches, both purely extractive:
  * the radiologist dictated a summary  -> reuse it (that is what the reference
    reports do), minus normal/negative filler;
  * no dictated summary                 -> condense the abnormal findings that
    were routed into FINDINGS and close with the template's normal impression.

Nothing is ever added that the dictation or template did not state.
"""
from __future__ import annotations

import re

from .dictation import POS_HINT
from .editor import NORMAL_STATE_RE, is_negative
from .template import resolve_placeholders
from .textutil import content_tokens, sim, squash, tidy_sentence

LEAD_EXISTENTIAL = re.compile(r"^there\s+(?:is|are|was|were)\s+(?:a|an|the)?\s*", re.I)
COPULA_TAIL = re.compile(
    r"\s+(?:is|are|was|were)\s+(?:identified|seen|noted|present|evident|visualized|"
    r"visualised|appreciated|demonstrated|observed)\b\.?$",
    re.I,
)
LEAD_ARTICLE = re.compile(r"^(?:a|an)\s+", re.I)
SPECIFICALLY = re.compile(r"^specifically,\s*", re.I)
# A template impression that asserts a normal study must not be appended next to
# abnormal findings - that would contradict the report.
TEMPLATE_NEGATIVE = re.compile(r"^\s*(no|without|negative for)\b", re.I)

DETAIL_TAIL = re.compile(
    r",?\s+(?:with|including|associated with|demonstrating|showing|producing|resulting in|"
    r"causing)\s+.+$",
    re.I,
)


def is_normal_statement(sentence: str) -> bool:
    return bool(NORMAL_STATE_RE.search(sentence)) and not POS_HINT.search(sentence)


def is_abnormal(sentence: str) -> bool:
    return (
        not is_negative(sentence)
        and not is_normal_statement(sentence)
        and bool(POS_HINT.search(sentence))
    )


def condense(text: str, trim_detail: bool = False) -> str:
    """Turn a findings sentence into an impression item (removal only)."""
    t = squash(text)
    t = LEAD_EXISTENTIAL.sub("", t)
    t = COPULA_TAIL.sub("", t)
    t = LEAD_ARTICLE.sub("", t)
    t = SPECIFICALLY.sub("", t)
    if trim_detail:
        head = DETAIL_TAIL.sub("", t)
        if len(content_tokens(head)) >= 3:
            t = head
    return tidy_sentence(t)


SEVERITY = (
    (re.compile(r"\b(severe|marked|extensive|large|complete|full-thickness|acute|"
                r"high-grade|gross|advanced)\b", re.I), 3.0),
    (re.compile(r"\b(moderate|moderate-to-severe|mild-to-moderate)\b", re.I), 2.0),
    (re.compile(r"\b(mild|minimal|small|trace|low-grade|early|subtle)\b", re.I), 1.0),
)


def severity(text: str) -> float:
    for pattern, weight in SEVERITY:
        if pattern.search(text):
            return weight
    return 1.5


def _dedupe(items: list[str], threshold: float = 0.82) -> list[str]:
    out: list[str] = []
    for it in items:
        if not squash(it):
            continue
        if any(sim(it, o) >= threshold for o in out):
            continue
        out.append(it)
    return out


def build_impression(
    dictated_summary: list[str],
    findings: list[str],
    template_impression: list[str],
    laterality: str | None,
    region: str | None,
    cfg,
) -> list[str]:
    tmpl_lines = [
        tidy_sentence(resolve_placeholders(x, laterality, region))
        for x in template_impression
        if squash(x)
    ]

    if dictated_summary:
        items = [condense(x) for x in dictated_summary]
        cap = cfg.summary_cap
    else:
        source = findings
        if cfg.findings_require_abnormal:
            abnormal = [x for x in findings if is_abnormal(x)]
            if not abnormal:
                # nothing abnormal was dictated: restating normal findings is
                # not an impression.  Fall back to the template's normal line,
                # optionally preceded by the dictation's own negative summary.
                if cfg.no_abnormal_fallback == "negatives":
                    source = [x for x in findings if is_negative(x)][-1:]
                else:
                    source = []
            else:
                source = abnormal
        items = [condense(x, trim_detail=cfg.trim_detail) for x in source]
        cap = cfg.findings_cap

    items = [x for x in items if not is_normal_statement(x)]
    if cfg.drop_negative_impression and not (
        not dictated_summary and cfg.no_abnormal_fallback == "negatives" and items
        and all(is_negative(x) for x in items)
    ):
        items = [x for x in items if not is_negative(x)]
    items = _dedupe(items, cfg.impression_dedupe)
    if cfg.rank_impression_by_severity and not dictated_summary:
        items = sorted(items, key=lambda x: -severity(x))
    if cap:
        items = items[:cap]

    if not dictated_summary and items and cfg.append_template_impression:
        closing = [x for x in tmpl_lines[:1] if TEMPLATE_NEGATIVE.match(x)]
        items = items + closing
    if not items:
        items = tmpl_lines or ["No acute abnormality."]

    if cfg.number_impression and len(items) > 1:
        return [f"{i}. {t}" for i, t in enumerate(items, 1)]
    return items


### 2.9 End-to-end pipeline

In [ ]:
%%writefile rrh/pipeline.py
"""End-to-end structured-generation pipeline.

    dictation ─▶ segment ─▶ split ─▶ route ─▶ edit template ─▶ impression ─▶ validate
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field as dc_field

from .dictation import segment_dictation
from .editor import edit_field, is_negative, render_clause
from .impression import build_impression
from .routing import (RoutingModel, build_context, cue_supported, field_candidates,
                      fit_ranked_router, score_segment)
from .splitting import candidate_splits
from .template import parse_template, render_report, resolve_placeholders
from .lexicon import stem
from .textutil import content_tokens, squash


@dataclass
class Config:
    # routing
    min_route_score: float = 0.45
    group_penalty: float = 1.0
    allow_splitting: bool = True
    cue_veto_penalty: float = 0.8
    w_cue: float = 3.0
    w_label: float = 2.2
    w_concept: float = 1.2
    w_template: float = 1.0
    w_mined: float = 1.2
    w_knn: float = 1.6
    w_continuity: float = 0.0
    w_backward: float = 0.0
    w_prior: float = 0.0
    viterbi: bool = False
    use_ranker: bool = False
    ranker_epochs: int = 300
    ranker_lr: float = 0.25
    ranker_l2: float = 0.001
    cue_match_min: float = 0.34
    split_margin: float = 0.0
    # editing
    cover_threshold: float = 0.12
    dedupe_threshold: float = 0.0  # 0 disables within-field de-duplication
    soften_blanket: bool = False
    add_copula: bool = False
    add_existential: bool = False
    abnormal_first: bool = False
    style_threshold: float = 0.0  # 0 disables reference-phrasing transfer
    correct_spelling: bool = True
    suppress_redundant_negatives: bool = False
    suppress_redundant_normals: bool = False
    normalize_shorthand: bool = True
    summary_threshold: float = 0.34
    summary_max_misses: int = 3
    summary_after_cues: bool = False
    recover_summary: bool = True
    summary_recover_threshold: float = 0.6
    # impression
    append_template_impression: bool = True
    number_impression: bool = True
    drop_negative_impression: bool = True
    trim_detail: bool = True
    findings_require_abnormal: bool = True
    no_abnormal_fallback: str = "template"  # template | negatives
    rank_impression_by_severity: bool = False
    summary_cap: int = 6
    impression_dedupe: float = 0.82
    findings_cap: int = 2
    # rendering
    blank_between_fields: bool = True
    keep_unrouted: bool = True
    merge_extras: bool = False


MODEL_KEYS = (
    "normalize_shorthand", "correct_spelling", "summary_threshold", "summary_max_misses",
    "use_ranker", "ranker_epochs", "ranker_lr", "ranker_l2", "summary_after_cues",
)


def model_key(cfg: "Config") -> str:
    """Identity of everything that changes the *fitted* model, for caching."""
    return "|".join(f"{k}={getattr(cfg, k)}" for k in MODEL_KEYS)


LATERAL_WORDS = {
    "rt": "right", "r": "right", "right": "right",
    "lt": "left", "l": "left", "left": "left",
    "bilateral": "bilateral", "b/l": "bilateral", "bilat": "bilateral", "both": "bilateral",
}
REGION_WORDS = {
    "lumbar spine": "lumbar", "lsspine": "lumbosacral", "thoracic spine": "thoracic",
    "cervical spine": "cervical", "spine sacrum": "sacral", "sacrum": "sacral",
}


def infer_laterality(row) -> str | None:
    for src in (row.get("study_description") or "", row.get("dictation") or ""):
        for m in re.finditer(r"\b(rt|lt|right|left|bilateral|bilat|b/l|both)\b", src, re.I):
            return LATERAL_WORDS.get(m.group(1).lower())
    return None


def infer_region(row) -> str | None:
    bp = (row.get("body_part") or "").strip().lower()
    if bp in REGION_WORDS:
        return REGION_WORDS[bp]
    sd = (row.get("study_description") or "").lower()
    for kw, val in (("lsp", "lumbar"), ("tsp", "thoracic"), ("csp", "cervical"),
                    ("lumbo", "lumbosacral"), ("lumbar", "lumbar"),
                    ("thoracic", "thoracic"), ("cervical", "cervical")):
        if kw in sd:
            return val
    return bp or None


@dataclass
class Trace:
    """What the pipeline decided - used by the validator and for auditing."""
    routed: dict[str, list[str]] = dc_field(default_factory=dict)
    extras: list[str] = dc_field(default_factory=list)
    dictated_impression: list[str] = dc_field(default_factory=list)
    abnormal: list[str] = dc_field(default_factory=list)
    unrouted_scores: list[tuple[str, float]] = dc_field(default_factory=list)
    normal_case: bool = False
    laterality: str | None = None
    region: str | None = None


class ReportGenerator:
    def __init__(self, model: RoutingModel, cfg: Config | None = None):
        self.model = model
        self.cfg = cfg or Config()
        c = self.cfg
        self._weights = {
            "cue": c.w_cue, "label": c.w_label, "concept": c.w_concept,
            "template": c.w_template, "mined": c.w_mined, "knn": c.w_knn,
            "continuity": c.w_continuity, "backward": c.w_backward, "prior": c.w_prior,
            "use_ranker": 1.0 if c.use_ranker else 0.0,
        }

    # -------------------------------------------------------------- routing
    def _route_one(self, text: str, cue: str | None, tmpl, ctx, prev_order=None):
        scored = score_segment(text, cue, tmpl, self.model, ctx, self._weights, prev_order)
        if not scored:
            return None, 0.0
        groups = {f.label for f in field_candidates(tmpl) if f.is_group}
        adjusted = [
            (s - (self.cfg.group_penalty if lab in groups else 0.0), lab) for s, lab in scored
        ]
        adjusted.sort(key=lambda x: (-x[0], x[1]))
        best_score, best_label = adjusted[0]
        if self.cfg.cue_veto_penalty and not cue_supported(
            cue, tmpl, ctx, self.cfg.cue_match_min
        ):
            best_score -= self.cfg.cue_veto_penalty
        if best_score < self.cfg.min_route_score:
            return None, best_score
        return best_label, best_score

    def _route_segment(self, text: str, cue: str | None, tmpl, ctx, prev_order=None):
        """Return list of (clause text, label|None, score)."""
        label, score = self._route_one(text, cue, tmpl, ctx, prev_order)
        whole = [(text, label, score)]
        if not self.cfg.allow_splitting:
            return whole
        for parts in candidate_splits(text):
            routed = [self._route_one(p, cue, tmpl, ctx, prev_order) for p in parts]
            labels = [lab for lab, _ in routed]
            if any(lab is None for lab in labels):
                continue
            if len(set(labels)) < 2:
                continue
            avg = sum(s for _, s in routed) / len(routed)
            if avg + self.cfg.split_margin >= score:
                return [(p, lab, s) for p, (lab, s) in zip(parts, routed)]
        return whole

    def _viterbi(self, clauses, tmpl, ctx):
        """Re-decode a fixed clause sequence as a path, not as independent picks.

        Emissions are the per-clause field scores; transitions encode the
        template-order structure of dictations (stay in the field, move on, or
        pay to jump backwards).  A NULL state absorbs clauses that belong to no
        field of this template.
        """
        cfg = self.cfg
        cands = field_candidates(tmpl)
        if not cands or not clauses:
            return [None] * len(clauses)
        groups = {f.label for f in cands if f.is_group}
        order = {f.label: f.order for f in cands}
        span = max(1, len(tmpl.fields) - 1)
        states = [f.label for f in cands] + [None]

        emissions = []
        for text, cue in clauses:
            scored = dict(
                (lab, sc) for sc, lab in score_segment(text, cue, tmpl, self.model, ctx,
                                                       self._weights)
            )
            penalty = (
                cfg.cue_veto_penalty
                if cfg.cue_veto_penalty
                and not cue_supported(cue, tmpl, ctx, cfg.cue_match_min)
                else 0.0
            )
            row = {}
            for lab in states:
                if lab is None:
                    row[lab] = cfg.min_route_score
                else:
                    row[lab] = (
                        scored.get(lab, 0.0)
                        - (cfg.group_penalty if lab in groups else 0.0)
                        - penalty
                    )
            emissions.append(row)

        def transition(prev, cur):
            if prev is None or cur is None:
                return 0.0
            a, b = order[prev], order[cur]
            if a == b:
                return cfg.w_continuity
            if b < a:
                return -cfg.w_backward * min(1.0, (a - b) / span)
            return 0.0

        best = {s: (emissions[0][s], [s]) for s in states}
        for row in emissions[1:]:
            nxt = {}
            for cur in states:
                score, path = max(
                    ((best[prev][0] + transition(prev, cur), best[prev][1]) for prev in states),
                    key=lambda x: x[0],
                )
                nxt[cur] = (score + row[cur], path + [cur])
            best = nxt
        return max(best.values(), key=lambda x: x[0])[1]

    # ------------------------------------------------------------- generate
    def generate(self, row: dict) -> tuple[str, Trace]:
        cfg = self.cfg
        tmpl = parse_template(row["template_content"])
        ctx = build_context(tmpl)
        doc = segment_dictation(
            row.get("dictation") or "",
            normalize=cfg.normalize_shorthand,
            vocab=self.model.vocab if cfg.correct_spelling else None,
            summary_threshold=cfg.summary_threshold,
            summary_max_misses=cfg.summary_max_misses,
            summary_after_cues=cfg.summary_after_cues,
        )
        laterality = infer_laterality(row)
        region = infer_region(row)
        trace = Trace(
            normal_case=doc.is_normal or not doc.findings,
            laterality=laterality,
            region=region,
        )

        routed: dict[str, list[str]] = {}
        extras: list[str] = []
        ordered_findings: list[str] = []
        units = list(doc.findings)
        if cfg.recover_summary and doc.impression:
            # A sentence in the dictated summary that restates nothing from the
            # body is not a summary at all - it is a finding, and must not be
            # lost just because it was dictated last.
            seen = {stem(t) for seg in doc.findings for t in content_tokens(seg.text)}
            for seg in doc.impression:
                st = {stem(t) for t in content_tokens(seg.text)}
                if st and len(st & seen) / len(st) < cfg.summary_recover_threshold:
                    units.append(seg)
        field_order = {f.label: f.order for f in tmpl.fields}
        prev_order: int | None = None
        for seg in units:
            for clause, label, score in self._route_segment(
                seg.text, seg.cue, tmpl, ctx, prev_order
            ):
                clause = squash(clause)
                if not clause:
                    continue
                if cfg.style_threshold:
                    styled = self.model.style_match(clause, cfg.style_threshold)
                    if styled:
                        clause = styled
                if label is None:
                    if cfg.keep_unrouted:
                        extras.append(clause)
                    trace.unrouted_scores.append((clause, score))
                else:
                    routed.setdefault(label, []).append(clause)
                    prev_order = field_order.get(label, prev_order)
                ordered_findings.append(clause)

        field_texts: list[tuple[str, str]] = []
        for f in tmpl.fields:
            if f.is_free:
                field_texts.append(("", resolve_placeholders(f.text, laterality, region)))
                continue
            if f.label == "OTHER FINDINGS":
                field_texts.append((f.label, ""))
                continue
            body = edit_field(f.text, routed.get(f.label, []), cfg)
            field_texts.append((f.label, resolve_placeholders(body, laterality, region)))

        impression = build_impression(
            [s.text for s in doc.impression],
            ordered_findings,
            tmpl.impression,
            laterality,
            region,
            cfg,
        )
        report = render_report(
            field_texts,
            impression,
            extra_paragraphs=[render_clause(e, cfg) for e in extras],
            blank_between_fields=cfg.blank_between_fields,
            merge_extras=cfg.merge_extras,
        )
        trace.routed = routed
        trace.extras = extras
        trace.dictated_impression = [s.text for s in doc.impression]
        trace.abnormal = [f for f in ordered_findings if not is_negative(f)]
        return report, trace


def build_generator(train_rows, cfg: Config | None = None) -> ReportGenerator:
    cfg = cfg or Config()
    return ReportGenerator(fit_ranked_router(train_rows, cfg), cfg)


### 2.10 Validation - negation, laterality, measurements, hallucination

In [ ]:
%%writefile rrh/validate.py
"""Validation layer.

Runs after generation and answers the questions the task brief asks for:
negation preserved, laterality preserved, measurements preserved, nothing
invented, untouched template fields untouched, dictated findings not dropped.

`validate` reports issues; `repair` fixes the ones that can be fixed
deterministically (a dropped finding is re-attached rather than lost).
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field as dc_field

from .dictation import LATERAL_RE, MEASURE_RE, segment_dictation
from .editor import is_negative
from .lexicon import stem
from .template import parse_template, resolve_placeholders
from .textutil import content_tokens, sim, split_sentences, squash


@dataclass
class Issue:
    kind: str
    detail: str
    severity: str = "warn"


@dataclass
class ValidationResult:
    issues: list[Issue] = dc_field(default_factory=list)

    def add(self, kind: str, detail: str, severity: str = "warn") -> None:
        self.issues.append(Issue(kind, detail, severity))

    @property
    def ok(self) -> bool:
        return not any(i.severity == "error" for i in self.issues)

    def counts(self) -> dict[str, int]:
        out: dict[str, int] = {}
        for i in self.issues:
            out[i.kind] = out.get(i.kind, 0) + 1
        return out


def _lateralities(text: str) -> set[str]:
    out = set()
    for m in LATERAL_RE.finditer(text or ""):
        v = m.group(1).lower()
        out.add({"rt": "right", "lt": "left", "b/l": "bilateral", "bilat": "bilateral"}.get(v, v))
    return out


def _measurements(text: str) -> set[str]:
    return {squash(m.group(0)).lower().replace(" ", "") for m in MEASURE_RE.finditer(text or "")}


LABEL_PREFIX = re.compile(r"^[ \t]*[A-Z][A-Z0-9 ,'\-/&\.\(\)]{0,58}?:[ \t]*", re.M)


def report_sentences(report: str) -> list[str]:
    """Sentences of a report with the field labels stripped off."""
    return split_sentences(LABEL_PREFIX.sub("", report))


def validate(row: dict, report: str, trace, vocab: dict | None = None) -> ValidationResult:
    res = ValidationResult()
    tmpl = parse_template(row["template_content"])
    out = parse_template(report)
    doc = segment_dictation(row.get("dictation") or "", vocab=vocab)

    # ---- structure -----------------------------------------------------
    if "FINDINGS:" not in report or "IMPRESSION:" not in report:
        res.add("structure", "missing FINDINGS/IMPRESSION header", "error")
    t_labels = [f.label for f in tmpl.fields if f.label]
    o_labels = [f.label for f in out.fields if f.label]
    if t_labels != o_labels:
        res.add("structure", f"label sequence changed: {set(t_labels) ^ set(o_labels)}", "error")
    if re.search(r"\[[^\]]*\]", report):
        res.add("placeholder", "unresolved [placeholder] left in report", "error")

    # ---- untouched fields ---------------------------------------------
    out_by_label = {f.label: squash(f.text) for f in out.fields if f.label}
    for f in tmpl.fields:
        if not f.label or f.label == "OTHER FINDINGS":
            continue
        if trace.routed.get(f.label):
            continue
        expected = squash(
            resolve_placeholders(f.text, trace.laterality, trace.region)
            if "[" in f.text
            else f.text
        )
        if out_by_label.get(f.label, "") != expected:
            res.add("untouched_field", f"{f.label} changed without a routed finding", "error")

    # ---- laterality ----------------------------------------------------
    sentences = report_sentences(report)
    for seg in doc.findings:
        lat = _lateralities(seg.text)
        if not lat:
            continue
        matched = [s for s in sentences if sim(seg.text, s) >= 0.5]
        if matched and not any(_lateralities(s) & lat for s in matched):
            res.add("laterality", f"laterality {sorted(lat)} lost: {seg.text[:70]}", "error")

    # ---- negation ------------------------------------------------------
    for seg in doc.findings:
        if not is_negative(seg.text):
            continue
        toks = {stem(t) for t in content_tokens(seg.text)}
        if not toks:
            continue
        matches = [
            s
            for s in sentences
            if (st := {stem(t) for t in content_tokens(s)}) and len(toks & st) / len(toks) >= 0.8
        ]
        if not matches:
            continue
        # the negation is preserved as long as *some* matching sentence still
        # states it negatively (the same phrase may also appear, correctly, as a
        # positive finding at another level or site)
        if not any(
            is_negative(s) or "without" in s.lower() or " no " in f" {s.lower()} "
            for s in matches
        ):
            res.add("negation", f"negated finding rendered positive: {seg.text[:70]}", "error")

    # ---- measurements --------------------------------------------------
    dict_meas = set()
    for seg in doc.findings + doc.impression:
        dict_meas |= _measurements(seg.text)
    lost = dict_meas - _measurements(report)
    for m in sorted(lost):
        res.add("measurement", f"measurement dropped: {m}")

    # ---- unsupported content (hallucination) ---------------------------
    allowed = set()
    normalised = " ".join(s.text for s in doc.preamble + doc.findings + doc.impression)
    for src in (row["template_content"], row.get("dictation") or "", normalised):
        allowed |= {stem(t) for t in content_tokens(src)}
    allowed |= {stem(t) for t in content_tokens(str(row.get("body_part") or ""))}
    allowed |= {stem(t) for t in content_tokens(str(row.get("study_description") or ""))}
    allowed |= {"remaining", "otherwise", "left", "right", "bilateral", "lumbar", "thoracic",
                "cervical", "lumbosacral", "sacral"}
    unseen = sorted({stem(t) for t in content_tokens(report)} - allowed)
    for t in unseen:
        res.add("unsupported_term", f"term not present in template or dictation: {t}", "error")

    # ---- omissions -----------------------------------------------------
    for seg in doc.findings:
        if len(content_tokens(seg.text)) < 2:
            continue
        toks = {stem(t) for t in content_tokens(seg.text)}
        rep_toks = {stem(t) for t in content_tokens(report)}
        if len(toks & rep_toks) / len(toks) < 0.6:
            res.add("omission", f"dictated finding not represented: {seg.text[:70]}")
    return res


def repair(row: dict, report: str, trace, result: ValidationResult) -> str:
    """Re-attach dictated findings that were dropped, before IMPRESSION."""
    missing = [i.detail.split(": ", 1)[1] for i in result.issues if i.kind == "omission"]
    if not missing:
        return report
    doc = segment_dictation(row.get("dictation") or "")
    texts = []
    for seg in doc.findings:
        if any(seg.text[:70] == m for m in missing):
            texts.append(squash(seg.text))
    if not texts:
        return report
    idx = report.find("IMPRESSION:")
    if idx < 0:
        return report
    block = "\n" + "\n\n".join(texts) + "\n\n"
    return report[:idx].rstrip("\n") + "\n" + block + report[idx:]


### 2.11 Offline scoring (RES proxies)

In [ ]:
%%writefile rrh/metrics.py
"""Offline scoring.

The leaderboard metric (RES - Radiology Edit Score, lower is better) is not
published, so we optimise a *family* of edit-based proxies and report them all.
`res_word` is the headline number used for tuning; `res_char` and the
structural scores guard against over-fitting one particular formulation.
"""
from __future__ import annotations

import re
from dataclasses import dataclass

from .template import parse_template
from .textutil import key, levenshtein, squash, token_set


def _words(text: str) -> list[str]:
    return key(text).split()


def res_word(pred: str, ref: str) -> float:
    r = _words(ref)
    p = _words(pred)
    if not r:
        return 0.0 if not p else 1.0
    return levenshtein(p, r) / len(r)


def res_char(pred: str, ref: str) -> float:
    r = squash(ref)
    p = squash(pred)
    if not r:
        return 0.0 if not p else 1.0
    return levenshtein(p, r) / len(r)


def res_sent(pred: str, ref: str) -> float:
    """Edit distance counted in whole sentences - the unit a radiologist edits."""
    from .textutil import split_sentences

    r = [key(x) for x in split_sentences(ref) if key(x)]
    p = [key(x) for x in split_sentences(pred) if key(x)]
    if not r:
        return 0.0 if not p else 1.0
    return levenshtein(p, r) / len(r)


def res_raw(pred: str, ref: str) -> float:
    """Formatting-sensitive variant: raw characters, nothing normalised."""
    r = (ref or "").strip()
    p = (pred or "").strip()
    if not r:
        return 0.0 if not p else 1.0
    return levenshtein(p, r) / len(r)


def field_scores(pred: str, ref: str) -> dict[str, float]:
    """Per-field agreement: did we edit the same fields, the same way?"""
    P = {f.label: squash(f.text) for f in parse_template(pred).fields if f.label}
    R = {f.label: squash(f.text) for f in parse_template(ref).fields if f.label}
    if not R:
        return {"field_label_f1": 1.0, "field_exact": 1.0, "field_res": 0.0}
    inter = set(P) & set(R)
    prec = len(inter) / len(P) if P else 0.0
    rec = len(inter) / len(R)
    f1 = 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)
    exact = sum(1 for k in inter if key(P[k]) == key(R[k])) / len(R)
    fres = sum(res_word(P.get(k, ""), R[k]) for k in R) / len(R)
    return {"field_label_f1": f1, "field_exact": exact, "field_res": fres}


_IMP = re.compile(r"^[ \t]*IMPRESSION[ \t]*:[ \t]*", re.I | re.M)


def split_parts(text: str) -> tuple[str, str]:
    m = _IMP.search(text)
    return (text[: m.start()], text[m.end() :]) if m else (text, "")


@dataclass
class Score:
    n: int
    res_word: float
    res_char: float
    res_raw: float
    res_sent: float
    findings_res: float
    impression_res: float
    field_label_f1: float
    field_exact: float
    field_res: float
    content_recall: float
    content_precision: float

    def as_row(self) -> str:
        return (
            f"n={self.n}  RES_word={self.res_word:.4f}  RES_char={self.res_char:.4f}  "
            f"RES_raw={self.res_raw:.4f}  RES_sent={self.res_sent:.4f}  "
            f"find={self.findings_res:.4f}  imp={self.impression_res:.4f}  "
            f"labelF1={self.field_label_f1:.4f}  fieldExact={self.field_exact:.4f}  "
            f"fieldRES={self.field_res:.4f}  cRec={self.content_recall:.3f}  "
            f"cPrec={self.content_precision:.3f}"
        )


def evaluate(preds: list[str], refs: list[str]) -> Score:
    acc = {
        k: 0.0
        for k in (
            "res_word res_char res_raw res_sent findings_res impression_res field_label_f1 "
            "field_exact field_res content_recall content_precision"
        ).split()
    }
    n = len(refs)
    for p, r in zip(preds, refs):
        acc["res_word"] += res_word(p, r)
        acc["res_char"] += res_char(p, r)
        acc["res_raw"] += res_raw(p, r)
        acc["res_sent"] += res_sent(p, r)
        pf, pi = split_parts(p)
        rf, ri = split_parts(r)
        acc["findings_res"] += res_word(pf, rf)
        acc["impression_res"] += res_word(pi, ri)
        for k, v in field_scores(p, r).items():
            acc[k] += v
        tp, tr_ = token_set(p), token_set(r)
        acc["content_recall"] += len(tp & tr_) / max(1, len(tr_))
        acc["content_precision"] += len(tp & tr_) / max(1, len(tp))
    return Score(n=n, **{k: v / max(1, n) for k, v in acc.items()})


## 3. Fit the routing model on the training set

In [ ]:
import importlib
import pandas as pd

for m in ["textutil", "lexicon", "template", "dictation", "splitting", "routing",
          "editor", "impression", "pipeline", "validate", "metrics"]:
    importlib.import_module(f"rrh.{m}")
    importlib.reload(sys.modules[f"rrh.{m}"])

from rrh.pipeline import Config, ReportGenerator
from rrh.routing import fit_router
from rrh.validate import validate

train = pd.read_csv(os.path.join(INPUT_DIR, "train.csv"))
test = pd.read_csv(os.path.join(INPUT_DIR, "test.csv"))
sample = pd.read_csv(os.path.join(INPUT_DIR, "sample_submission.csv"))
print(train.shape, test.shape, sample.shape)

CONFIG = json.loads("""
{
  "abnormal_first": true,
  "add_copula": false,
  "add_existential": false,
  "allow_splitting": true,
  "append_template_impression": true,
  "blank_between_fields": true,
  "correct_spelling": true,
  "cover_threshold": 0.05,
  "cue_match_min": 0.34,
  "cue_veto_penalty": 1.4,
  "dedupe_threshold": 0.7,
  "drop_negative_impression": false,
  "findings_cap": 2,
  "findings_require_abnormal": true,
  "group_penalty": 1.0,
  "impression_dedupe": 0.95,
  "keep_unrouted": true,
  "min_route_score": 0.2,
  "normalize_shorthand": true,
  "number_impression": true,
  "rank_impression_by_severity": false,
  "recover_summary": false,
  "soften_blanket": false,
  "split_margin": -0.3,
  "style_threshold": 0.0,
  "summary_after_cues": false,
  "summary_cap": 8,
  "summary_max_misses": 2,
  "summary_recover_threshold": 0.6,
  "summary_threshold": 0.23,
  "suppress_redundant_negatives": false,
  "suppress_redundant_normals": false,
  "trim_detail": true,
  "w_backward": 1.0,
  "w_continuity": 0.3,
  "w_cue": 2.0,
  "w_template": 1.6
}
""")
cfg = Config(**CONFIG)
generator = ReportGenerator(fit_router(train.to_dict("records")), cfg)
print(json.dumps(CONFIG, indent=2, sort_keys=True))


## 4. Cross-validated offline score

RES is not published, so we track a family of edit-based proxies (word/char normalised edit distance to the reference, plus field-level fidelity, content recall and precision). Lower is better.

In [ ]:
from rrh.metrics import evaluate
from rrh.pipeline import ReportGenerator

# Honest cross-validation: the routing statistics are re-mined per fold so no
# fold ever sees its own reference report.
K = 5
rows = train.to_dict("records")
folds = [rows[i::K] for i in range(K)]
preds, gold = [], []
for f in range(K):
    tr_rows = [r for j in range(K) if j != f for r in folds[j]]
    g = ReportGenerator(fit_router(tr_rows), cfg)
    for r in folds[f]:
        preds.append(g.generate(r)[0])
        gold.append(r["report"])

from rrh.template import parse_template, render_report
baseline = []
for r in rows:
    t = parse_template(r["template_content"])
    baseline.append(render_report([(x.label, x.text) for x in t.fields], t.impression))

print("copy-the-template baseline :", evaluate(baseline, [r["report"] for r in rows]).as_row())
print("structured pipeline (5-CV) :", evaluate(preds, gold).as_row())


## 5. Generate `submission.csv`

In [ ]:
reports, issue_counts = {}, {}
for row in test.to_dict("records"):
    report, trace = generator.generate(row)
    reports[row["case_id"]] = report.strip()
    for k, v in validate(row, report, trace).counts().items():
        issue_counts[k] = issue_counts.get(k, 0) + v

order = sample["case_id"].tolist()
assert set(order) == set(reports), "case_id set differs from sample_submission"
assert len(order) == len(set(order)) == len(test) == 132, "row count / duplicate check failed"

with open("submission.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh, quoting=csv.QUOTE_ALL, lineterminator="\n")
    w.writerow(["case_id", "report"])
    for cid in order:
        w.writerow([cid, reports[cid]])

print("submission.csv rows:", len(order))
print("validation issues  :", json.dumps(issue_counts, sort_keys=True) or "none")
print()
print(reports[order[0]])


## 6. Optional hybrid LLM pass (off by default, no keys in the notebook)

In [ ]:
"""OPTIONAL: LLM refinement pass (disabled by default).

The submitted `submission.csv` is produced by the deterministic pipeline above.
This cell shows how the same pipeline can be run as a hybrid (Approach D):
the deterministic editor proposes the report, and a hosted model is asked only
to *re-word* clauses it already contains - never to add findings.

No API key is stored in this notebook.  It is read from an environment
variable (Kaggle: Add-ons -> Secrets).  With no key present the cell is a
no-op, so the notebook still reproduces the submission exactly.
"""
API_KEY = os.environ.get("ANTHROPIC_API_KEY")  # or a Kaggle Secret of the same name
USE_LLM = bool(API_KEY) and os.environ.get("RRH_USE_LLM") == "1"

SYSTEM_PROMPT = """You are a precision report editor, not a radiologist.
You receive a normal template, a dictation, and a draft report built by editing
that template. Return the draft with wording corrections only.
Rules:
- Never add a finding, diagnosis, measurement or laterality that is not in the dictation.
- Never remove a dictated finding.
- Keep every field label and the field order exactly as in the template.
- Leave fields the dictation does not mention byte-identical to the template.
- Output only FINDINGS: ... IMPRESSION: ..."""

def refine(row, draft):
    if not USE_LLM:
        return draft
    import anthropic
    client = anthropic.Anthropic(api_key=API_KEY)
    msg = client.messages.create(
        model="claude-opus-5",
        max_tokens=2000,
        temperature=0,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content":
                   f"TEMPLATE:\n{row['template_content']}\n\n"
                   f"DICTATION:\n{row['dictation']}\n\nDRAFT:\n{draft}"}],
    )
    return msg.content[0].text.strip()

print("LLM refinement enabled:", USE_LLM)
